# Reproducible benchmark - Phase 2

The frozen Phase 1 pipeline on measured, real data. `tools/build_benchmark.py`
assembles reference/target pairs from COCO 2017 val (category-level transfer,
one pair per object, no artificial viewpoint, illumination, scale, occlusion,
noise or blur transforms), `evaluation/dataset_integrity.py` gates the data
before anything runs, and `tools/run_benchmark.py` scores every pair with
per-sample resume and full provenance.

| step | module | output |
| --- | --- | --- |
| 1. pairs | `tools/build_benchmark.py` | `data/samples/`, `data/splits/` |
| 2. gate | `evaluation/dataset_integrity.py` | PASS or findings |
| 3. run | `tools/run_benchmark.py` | `results.csv`, `summary.json` |
| 4. figures | `visualization/benchmark.py` | IoU, group and runtime plots |

Dataset choices, citations and bandwidth measurements are documented in
`docs/dataset_comparison.md`; the selection rules in `docs/pairing_protocol.md`.

Generated from source commit content, fingerprint `6e041d5bbedfb2d5`. Re-run
`python tools/embed_sources.py` in VS Code after changing the modules.

## 1. Environment

In [ ]:
!pip -q install "transformers>=4.40" accelerate safetensors 2>/dev/null || pip -q install "transformers>=4.40" accelerate safetensors

import importlib
import subprocess
import sys

for package in ("torch", "transformers", "numpy", "scipy", "matplotlib", "PIL"):
    try:
        module = importlib.import_module(package)
        print(f"{package:14s} {getattr(module, '__version__', 'ok')}")
    except ImportError:
        print(f"{package:14s} MISSING - install it before continuing")

try:
    import torch

    print("cuda available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as error:
    print("torch check failed:", error)

## Project source

`# Generated by tools/embed_sources.py - do not edit this file directly.`

In [ ]:
import os
import sys

BUNDLE_ROOT = "one_shot_src"
EXPECTED_SOURCE_REVISION = "6e041d5bbedfb2d5"
SOURCE_FILES = {
    "evaluation/cases.py": "\"\"\"Evaluation cases: the bridge between a dataset on disk and the pipeline.\n\nA few-shot case is a reference image plus its mask, a target image, and\noptionally a target mask to score against. This module stays deliberately\nformat-agnostic: it does not know how COCO-20i, FSS-1000, LVIS-92i, PACO-Part or\nPascal-Part encode their splits, because those come from external archives whose\ninternal layout is not ours to assume. Instead, a case is described by paths, and\n`cases_from_manifest` reads that description from JSON:\n\n    {\n      \"root\": \"datasets\",\n      \"cases\": [\n        {\"name\": \"coco-20i-val-0001\",\n         \"reference_image\": \"COCO2014/val2014/COCO_val2014_000000391895.jpg\",\n         \"reference_mask\":  \"COCO2014/annotations/val2014/000000391895.png\",\n         \"target_image\":    \"COCO2014/val2014/COCO_val2014_000000000139.jpg\",\n         \"target_mask\":     \"COCO2014/annotations/val2014/000000000139.png\"}\n      ]\n    }\n\nMultiple instances in one mask file are supported through `instance_index`, which\nkeeps only the requested label, so a converted COCO-20i mask file and a\nsingle-object Pascal-Part mask both work unchanged.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport json\nimport os\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom evaluation.metrics import EvaluationReport, evaluate_results\nfrom matching.reference_representation import MaskInput\nfrom models.feature_encoder import ImageInput, to_pil_image\n\n\n@dataclass\nclass Case:\n    \"\"\"One few-shot example.\n\n    Attributes:\n        name: identifier used in the report table.\n        reference_image: image containing the query object.\n        reference_mask: binary mask of that object.\n        target_image: image to segment.\n        target_mask: optional ground truth for scoring.\n        meta: anything else worth keeping (category, dataset, split).\n    \"\"\"\n\n    name: str\n    reference_image: ImageInput\n    reference_mask: MaskInput\n    target_image: ImageInput\n    target_mask: Optional[MaskInput] = None\n    meta: Dict[str, Any] = field(default_factory=dict)\n\n    @property\n    def scorable(self) -> bool:\n        return self.target_mask is not None\n\n    def describe(self) -> Dict[str, Any]:\n        return {\"name\": self.name, \"scorable\": self.scorable, **self.meta}\n\n\ndef load_mask(path: str, instance_index: Optional[int] = None) -> np.ndarray:\n    \"\"\"Read a mask image; `instance_index` keeps only that instance's label.\n\n    White foreground over black background is assumed. A multi-colour palette\n    file is handled by treating the top-left pixel as the background and every\n    distinct remaining colour as one instance.\n    \"\"\"\n    from PIL import Image\n\n    array = np.asarray(Image.open(path))\n    if array.ndim == 3:\n        background = array[0, 0].astype(np.int16)\n        mask = np.abs(array.astype(np.int16) - background).sum(axis=-1) > 24\n        if instance_index is not None:\n            colours = np.unique(array[mask]) if mask.any() else np.zeros((0, array.shape[-1]), array.dtype)\n            if instance_index >= len(colours):\n                raise IndexError(\n                    f\"{path} holds {len(colours)} instance labels, cannot take index {instance_index}\"\n                )\n            mask = (array == colours[instance_index]).all(axis=-1)\n    else:\n        if instance_index is not None:\n            raise ValueError(f\"{path} is a single-instance grayscale mask; drop instance_index\")\n        mask = array > 127\n    return mask.astype(bool)\n\n\ndef case_from_paths(\n    root: str,\n    name: str,\n    reference_image: str,\n    reference_mask: str,\n    target_image: str,\n    target_mask: Optional[str] = None,\n    instance_index: Optional[int] = None,\n    target_instance_index: Optional[int] = None,\n    meta: Optional[Dict[str, Any]] = None,\n) -> Case:\n    \"\"\"Build a `Case` from paths relative to `root`.\"\"\"\n    def resolve(relative: Optional[str]) -> Optional[str]:\n        if relative is None:\n            return None\n        path = relative if os.path.isabs(relative) else os.path.join(root, relative)\n        if not os.path.exists(path):\n            raise FileNotFoundError(path)\n        return path\n\n    reference_path = resolve(reference_image)\n    target_path = resolve(target_image)\n    assert reference_path is not None and target_path is not None\n    return Case(\n        name=name,\n        reference_image=to_pil_image(reference_path),\n        reference_mask=load_mask(resolve(reference_mask) or \"\", instance_index),\n        target_image=to_pil_image(target_path),\n        target_mask=None if target_mask is None else load_mask(resolve(target_mask) or \"\", target_instance_index),\n        meta=dict(meta or {}),\n    )\n\n\ndef cases_from_manifest(path: str, root: Optional[str] = None) -> List[Case]:\n    \"\"\"Read a JSON manifest describing several cases.\"\"\"\n    with open(path, \"r\", encoding=\"utf-8\") as handle:\n        manifest = json.load(handle)\n    base = root or manifest.get(\"root\") or os.path.dirname(os.path.abspath(path))\n    cases: List[Case] = []\n    for entry in manifest.get(\"cases\", []):\n        meta = {key: value for key, value in entry.items() if key not in {\n            \"name\", \"reference_image\", \"reference_mask\", \"target_image\", \"target_mask\",\n            \"instance_index\", \"target_instance_index\"}}\n        cases.append(\n            case_from_paths(\n                base,\n                entry[\"name\"],\n                entry[\"reference_image\"],\n                entry[\"reference_mask\"],\n                entry[\"target_image\"],\n                entry.get(\"target_mask\"),\n                entry.get(\"instance_index\"),\n                entry.get(\"target_instance_index\"),\n                meta,\n            )\n        )\n    return cases\n\n\ndef run_cases(pipeline: Any, cases: Sequence[Case], verbose: bool = True) -> EvaluationReport:\n    \"\"\"Run every case through the pipeline and score the ones with a ground truth.\"\"\"\n    from evaluation.metrics import evaluate_mask\n\n    rows: List[Tuple[str, Optional[np.ndarray], Optional[np.ndarray], Any]] = []\n    for case in cases:\n        result = pipeline.segment_object(case.reference_image, case.reference_mask, case.target_image)\n        rows.append((case.name, result.predicted_mask, case.target_mask, result))\n        if verbose:\n            suffix = \"\" if not case.scorable else f\" IoU {evaluate_mask(result.predicted_mask, case.target_mask).iou:.3f}\"\n            print(f\"{case.name:32s} conf {result.confidence:.3f} \"\n                  f\"{result.runtime['total']:.2f}s{suffix}\")\n    scorable = [row for row in rows if row[2] is not None]\n    if not scorable:\n        return EvaluationReport(\n            rows=[{\"example\": name, \"confidence\": float(result.confidence),\n                   \"seconds\": float(result.runtime[\"total\"])}\n                  for name, _, _, result in rows],\n            metric_names=(\"confidence\",),\n        )\n    return evaluate_results(scorable)\n",
    "evaluation/dataset_integrity.py": "\"\"\"Integrity checks for a benchmark laid out on disk.\n\nA benchmark result is worthless if a mask was silently empty, an image was\ntruncated by a failed download, or two samples share an identifier, so the\nchecks live here as a plain function that a CLI, a notebook, and a test can all\ncall. Findings carry a severity so that a report can escalate warnings without\ncode changes:\n\n`critical`\n    The sample cannot be scored as stored: a missing file, an unreadable image, a\n    mask that is empty or does not match its image. A benchmark run must stop.\n`warning`\n    The sample is scorable but suspicious: metadata disagreeing with the pixels, a\n    photograph reused across samples, or a pair repeated across splits. Raise\n    these to critical with `--strict`.\n`info`\n    Facts worth recording, such as how the dataset is spread over categories.\n\nThe checker never modifies the dataset and never fetches anything.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport json\nimport os\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, List, Optional, Sequence, Set, Tuple\n\nimport numpy as np\n\nCRITICAL = \"critical\"\nWARNING = \"warning\"\nINFO = \"info\"\n\nREQUIRED_FILES = (\"reference.jpg\", \"reference_mask.png\", \"target.jpg\", \"target_mask.png\",\n                  \"metadata.json\")\nROLES = (\"reference\", \"target\")\nCONDITION_VALUES = {\n    \"viewpoint_condition\": {\"unknown\"},\n    \"illumination_condition\": {\"unknown\"},\n    \"occlusion_condition\": {\"unknown\"},\n    \"scale_condition\": {\"similar\", \"smaller_in_target\", \"larger_in_target\"},\n}\nMETADATA_KEYS = (\"sample_id\", \"object_id\", \"dataset\", \"category\") + tuple(CONDITION_VALUES)\nBBOX_TOLERANCE_PX = 3.0\n\n\n@dataclass\nclass Finding:\n    \"\"\"One problem or observation about one sample.\"\"\"\n\n    severity: str\n    code: str\n    sample_id: str\n    message: str\n\n    def to_dict(self) -> Dict[str, str]:\n        return {\"severity\": self.severity, \"code\": self.code,\n                \"sample_id\": self.sample_id, \"message\": self.message}\n\n\n@dataclass\nclass IntegrityReport:\n    \"\"\"Everything the checker found, plus a few counts.\"\"\"\n\n    root: str\n    findings: List[Finding] = field(default_factory=list)\n    samples_checked: int = 0\n    categories: Counter = field(default_factory=Counter)\n    splits: Dict[str, List[str]] = field(default_factory=dict)\n\n    def add(self, severity: str, code: str, sample_id: str, message: str) -> None:\n        self.findings.append(Finding(severity, code, sample_id, message))\n\n    def counts(self) -> Dict[str, int]:\n        return dict(Counter(finding.severity for finding in self.findings))\n\n    @property\n    def critical(self) -> List[Finding]:\n        return [finding for finding in self.findings if finding.severity == CRITICAL]\n\n    def ok(self) -> bool:\n        return not self.critical\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            \"root\": self.root,\n            \"samples_checked\": self.samples_checked,\n            \"counts\": self.counts(),\n            \"ok\": self.ok(),\n            \"categories\": dict(sorted(self.categories.items())),\n            \"splits\": {name: len(items) for name, items in sorted(self.splits.items())},\n            \"findings\": [finding.to_dict() for finding in self.findings],\n        }\n\n    def summary(self, limit: int = 12) -> str:\n        \"\"\"Format the report for a terminal.\"\"\"\n        counts = self.counts()\n        lines = [\n            f\"dataset root: {self.root}\",\n            f\"samples: {self.samples_checked} | categories: {len(self.categories)}\",\n            \"splits: \" + \", \".join(f\"{name} {len(items)}\" for name, items in sorted(self.splits.items())),\n            \"findings: \" + \", \".join(f\"{severity} {count}\" for severity, count in sorted(counts.items()))\n            or \"findings: none\",\n        ]\n        for severity in (CRITICAL, WARNING):\n            shown = [finding for finding in self.findings if finding.severity == severity][:limit]\n            for finding in shown:\n                lines.append(f\"  [{severity}] {finding.code} {finding.sample_id}: {finding.message}\")\n            hidden = len([f for f in self.findings if f.severity == severity]) - len(shown)\n            if hidden > 0:\n                lines.append(f\"  [{severity}] ... {hidden} more\")\n        lines.append(\"RESULT: \" + (\"PASS\" if self.ok() else \"FAIL\"))\n        return \"\\n\".join(lines)\n\n\ndef _open_image(path: str) -> Any:\n    from PIL import Image\n\n    image = Image.open(path)\n    image.load()\n    return image\n\n\ndef _mask_bbox(mask: np.ndarray) -> Optional[Tuple[float, float, float, float]]:\n    rows, columns = np.nonzero(mask)\n    if rows.size == 0:\n        return None\n    return (float(columns.min()), float(rows.min()),\n            float(columns.max() - columns.min() + 1), float(rows.max() - rows.min() + 1))\n\n\ndef check_sample(directory: str, report: IntegrityReport) -> Optional[Dict[str, Any]]:\n    \"\"\"Validate one sample folder, recording findings against `report`.\"\"\"\n    name = os.path.basename(os.path.normpath(directory))\n    for filename in REQUIRED_FILES:\n        if not os.path.isfile(os.path.join(directory, filename)):\n            report.add(CRITICAL, \"missing_file\", name, f\"{filename} is absent\")\n    if report.critical and any(finding.sample_id == name for finding in report.critical):\n        return None\n\n    try:\n        with open(os.path.join(directory, \"metadata.json\"), \"r\", encoding=\"utf-8\") as handle:\n            metadata = json.load(handle)\n    except (OSError, json.JSONDecodeError) as error:\n        report.add(CRITICAL, \"corrupt_metadata\", name, f\"metadata.json unreadable: {error}\")\n        return None\n\n    sample_id = str(metadata.get(\"sample_id\", name))\n    for key in METADATA_KEYS:\n        if key not in metadata:\n            report.add(CRITICAL, \"metadata_field\", sample_id, f\"metadata lacks {key}\")\n    for key, allowed in CONDITION_VALUES.items():\n        value = metadata.get(key)\n        if key in metadata and value not in allowed:\n            report.add(WARNING, \"condition_value\", sample_id,\n                       f\"{key}={value!r} is not one of {sorted(allowed)}\")\n\n    measured = metadata.get(\"measured\", {}) if isinstance(metadata.get(\"measured\"), dict) else {}\n    for role in ROLES:\n        image_path = os.path.join(directory, f\"{role}.jpg\")\n        mask_path = os.path.join(directory, f\"{role}_mask.png\")\n        try:\n            image = _open_image(image_path)\n        except Exception as error:  # noqa: BLE001 - any decode failure means the file is unusable\n            report.add(CRITICAL, \"corrupt_image\", sample_id, f\"{role}.jpg unreadable: {error}\")\n            continue\n        try:\n            mask_image = _open_image(mask_path)\n        except Exception as error:  # noqa: BLE001\n            report.add(CRITICAL, \"corrupt_mask\", sample_id, f\"{role}_mask.png unreadable: {error}\")\n            continue\n\n        mask = np.asarray(mask_image)\n        if mask.ndim == 3:\n            mask = mask[..., 0]\n        binary = mask > 127\n        if not binary.any():\n            report.add(CRITICAL, \"empty_mask\", sample_id, f\"{role} mask has no foreground\")\n            continue\n        if mask_image.size != image.size:\n            report.add(CRITICAL, \"size_mismatch\", sample_id,\n                       f\"{role} mask is {mask_image.size} but the image is {image.size}\")\n            continue\n        expected_fraction = measured.get(f\"{role}_area_fraction\")\n        if isinstance(expected_fraction, (int, float)):\n            if abs(float(binary.mean()) - float(expected_fraction)) > 0.01:\n                report.add(WARNING, \"area_mismatch\", sample_id,\n                           f\"{role} mask covers {binary.mean():.4f} of the image but metadata \"\n                           f\"says {float(expected_fraction):.4f}\")\n        expected_bbox = measured.get(f\"{role}_bbox\")\n        if isinstance(expected_bbox, (list, tuple)) and len(expected_bbox) == 4:\n            box = _mask_bbox(binary)\n            if box is None:\n                report.add(CRITICAL, \"empty_mask\", sample_id, f\"{role} mask has no foreground\")\n            else:\n                deviation = max(abs(box[index] - float(expected_bbox[index])) for index in range(4))\n                if deviation > BBOX_TOLERANCE_PX:\n                    report.add(WARNING, \"bbox_mismatch\", sample_id,\n                               f\"{role} mask box deviates {deviation:.1f}px from the source \"\n                               f\"annotation box\")\n    return metadata\n\n\ndef check_dataset(root: str = \"data\", strict: bool = False,\n                  samples_dir: Optional[str] = None) -> IntegrityReport:\n    \"\"\"Check every sample folder under `root` and cross-check the split manifests.\n\n    Args:\n        root: dataset root holding `samples/`, `splits/`, and optionally `raw/`.\n        strict: promote warnings to critical, for a pre-publication check.\n        samples_dir: override the samples directory, for a root with a different\n            layout.\n\n    Returns:\n        An `IntegrityReport`. `report.ok()` is false when anything critical was\n        found, which is the signal a benchmark runner should stop on.\n    \"\"\"\n    report = IntegrityReport(root=root)\n    directory = samples_dir or os.path.join(root, \"samples\")\n    if not os.path.isdir(directory):\n        report.add(CRITICAL, \"missing_samples_dir\", \"-\", f\"{directory} does not exist\")\n        return report\n\n    names = sorted(entry for entry in os.listdir(directory)\n                   if os.path.isdir(os.path.join(directory, entry)))\n    seen_sample_ids: Dict[str, str] = {}\n    seen_pairs: Dict[Tuple[str, str], str] = {}\n    image_usage: Dict[str, List[str]] = {}\n\n    for name in names:\n        metadata = check_sample(os.path.join(directory, name), report)\n        if metadata is None:\n            continue\n        report.samples_checked += 1\n        sample_id = str(metadata.get(\"sample_id\", name))\n        category = str(metadata.get(\"category\", \"unknown\"))\n        report.categories[category] += 1\n\n        if sample_id in seen_sample_ids:\n            report.add(CRITICAL, \"duplicate_sample_id\", sample_id,\n                       f\"also used by {seen_sample_ids[sample_id]}\")\n        seen_sample_ids[sample_id] = name\n        if sample_id != name:\n            report.add(WARNING, \"id_folder_mismatch\", sample_id,\n                       f\"stored in folder {name}\")\n\n        source = metadata.get(\"source\", {}) if isinstance(metadata.get(\"source\"), dict) else {}\n        image_ids = source.get(\"image_id\") or []\n        if isinstance(image_ids, list) and len(image_ids) == 2:\n            key = (str(image_ids[0]), str(image_ids[1]))\n            pair_key = tuple(sorted(key))\n            if pair_key in seen_pairs:\n                report.add(WARNING, \"duplicate_pair\", sample_id,\n                           f\"same two photographs as {seen_pairs[pair_key]}\")\n            seen_pairs[pair_key] = sample_id\n            for value in key:\n                image_usage.setdefault(value, []).append(sample_id)\n\n    for photograph, owners in sorted(image_usage.items()):\n        unique = sorted(set(owners))\n        if len(unique) > 2:\n            report.add(INFO, \"image_reuse\", unique[0],\n                       f\"source photograph {photograph} appears in {len(unique)} samples\")\n\n    splits_dir = os.path.join(root, \"splits\")\n    if os.path.isdir(splits_dir):\n        split_pairs: Dict[Tuple[str, str], str] = {}\n        for filename in sorted(os.listdir(splits_dir)):\n            if not filename.endswith(\".json\"):\n                continue\n            split = filename[:-5]\n            path = os.path.join(splits_dir, filename)\n            try:\n                with open(path, \"r\", encoding=\"utf-8\") as handle:\n                    manifest = json.load(handle)\n            except (OSError, json.JSONDecodeError) as error:\n                report.add(CRITICAL, \"corrupt_split\", split, f\"unreadable: {error}\")\n                continue\n            entries = manifest.get(\"cases\", [])\n            report.splits[split] = [str(entry.get(\"name\", \"?\")) for entry in entries]\n            on_disk: Set[str] = set()\n            for entry in entries:\n                case_name = str(entry.get(\"name\", \"?\"))\n                for role in ROLES:\n                    relative = entry.get(f\"{role}_image\")\n                    if not isinstance(relative, str):\n                        report.add(CRITICAL, \"split_field\", case_name,\n                                   f\"manifest entry lacks {role}_image\")\n                        continue\n                    resolved = os.path.normpath(os.path.join(splits_dir, relative))\n                    if not os.path.isfile(resolved):\n                        report.add(CRITICAL, \"split_path\", case_name,\n                                   f\"{role}_image does not resolve: {relative}\")\n                    else:\n                        on_disk.add(os.path.basename(os.path.dirname(resolved)))\n            missing = sorted(set(report.splits[split]) - on_disk)\n            if missing:\n                report.add(WARNING, \"split_coverage\", split,\n                           f\"{len(missing)} listed cases were not found on disk \"\n                           f\"(first: {missing[0]})\")\n            for entry in entries:\n                image = entry.get(\"reference_image\")\n                target = entry.get(\"target_image\")\n                if isinstance(image, str) and isinstance(target, str):\n                    key = tuple(sorted((os.path.normpath(image), os.path.normpath(target))))\n                    if key in split_pairs and split_pairs[key] != split:\n                        report.add(WARNING, \"split_leakage\", str(entry.get(\"name\", \"?\")),\n                                   f\"pair also appears in split {split_pairs[key]}\")\n                    split_pairs.setdefault(key, split)\n    else:\n        report.add(WARNING, \"missing_splits_dir\", \"-\", f\"{splits_dir} does not exist\")\n\n    if strict:\n        for finding in report.findings:\n            if finding.severity == WARNING:\n                finding.severity = CRITICAL\n    return report\n\n\ndef main(argv: Optional[Sequence[str]] = None) -> int:\n    import argparse\n\n    parser = argparse.ArgumentParser(description=\"Verify benchmark data integrity\",\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument(\"--root\", default=\"data\", help=\"dataset root (default: data)\")\n    parser.add_argument(\"--samples-dir\", default=None, help=\"override the samples directory\")\n    parser.add_argument(\"--strict\", action=\"store_true\", help=\"treat warnings as failures\")\n    parser.add_argument(\"--json\", default=None, help=\"write the full report to this path\")\n    parser.add_argument(\"--max-shown\", type=int, default=12,\n                        help=\"findings printed per severity (default: 12)\")\n    arguments = parser.parse_args(argv)\n\n    report = check_dataset(root=arguments.root, strict=arguments.strict,\n                           samples_dir=arguments.samples_dir)\n    print(report.summary(limit=arguments.max_shown))\n    if arguments.json:\n        os.makedirs(os.path.dirname(os.path.abspath(arguments.json)), exist_ok=True)\n        with open(arguments.json, \"w\", encoding=\"utf-8\") as handle:\n            json.dump(report.to_dict(), handle, indent=2)\n            handle.write(\"\\n\")\n        print(f\"report -> {arguments.json}\")\n    return 0 if report.ok() else 1\n\n\nif __name__ == \"__main__\":\n    import sys\n\n    sys.exit(main())\n",
    "evaluation/metrics.py": "\"\"\"Step 9 - evaluation metrics for a small, verified set of examples.\n\nThe four metrics requested for Phase 1 are IoU, Dice, precision and recall.\nThey are computed from boolean masks, tolerate empty inputs by returning\ndefined values (1.0 when both sides are empty, 0.0 when only the prediction is\nempty), and are paired with a `summarise` helper that reports the mean plus the\nper-example rows, so a handful of results stays readable.\n\nPhase 1 evaluates a handful of manually checked examples, so this module\ndeliberately has no dataset iteration, no sweeps and no aggregation tricks.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nfrom dataclasses import dataclass\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom matching.reference_representation import MaskInput, to_binary_mask\n\nEPSILON = 1e-7\n\n\n@dataclass\nclass MaskMetrics:\n    \"\"\"Overlap metrics for one prediction against one ground-truth mask.\"\"\"\n\n    iou: float\n    dice: float\n    precision: float\n    recall: float\n    true_positive: int\n    false_positive: int\n    false_negative: int\n\n    def to_dict(self) -> Dict[str, float]:\n        return {\n            \"iou\": self.iou,\n            \"dice\": self.dice,\n            \"precision\": self.precision,\n            \"recall\": self.recall,\n            \"true_positive\": float(self.true_positive),\n            \"false_positive\": float(self.false_positive),\n            \"false_negative\": float(self.false_negative),\n        }\n\n    def __str__(self) -> str:\n        return f\"IoU {self.iou:.3f} | Dice {self.dice:.3f} | P {self.precision:.3f} | R {self.recall:.3f}\"\n\n\ndef _as_bool(mask: Optional[MaskInput]) -> np.ndarray:\n    if mask is None:\n        return np.zeros((0, 0), dtype=bool)\n    tensor = to_binary_mask(mask)\n    return (tensor.squeeze() > 0.5).cpu().numpy().astype(bool)\n\n\ndef evaluate_mask(prediction: Optional[MaskInput], ground_truth: MaskInput) -> MaskMetrics:\n    \"\"\"IoU, Dice, precision and recall for one predicted mask.\"\"\"\n    predicted = _as_bool(prediction)\n    truth = _as_bool(ground_truth)\n\n    if predicted.shape != truth.shape:\n        if predicted.size == 0:\n            predicted = np.zeros(truth.shape, dtype=bool)\n        else:\n            import torch.nn.functional as F\n\n            resized = F.interpolate(\n                to_binary_mask(prediction), size=truth.shape[-2:], mode=\"nearest\"\n            )\n            predicted = (resized.squeeze() > 0.5).cpu().numpy().astype(bool)\n\n    intersection = int(np.logical_and(predicted, truth).sum())\n    predicted_area = int(predicted.sum())\n    truth_area = int(truth.sum())\n    union = predicted_area + truth_area - intersection\n\n    if union == 0:\n        iou = 1.0\n        dice = 1.0\n    else:\n        iou = intersection / union\n        dice = (2.0 * intersection) / (predicted_area + truth_area) if (predicted_area + truth_area) else 0.0\n\n    precision = intersection / predicted_area if predicted_area else (1.0 if truth_area == 0 else 0.0)\n    recall = intersection / truth_area if truth_area else (1.0 if predicted_area == 0 else 0.0)\n\n    return MaskMetrics(\n        iou=float(iou),\n        dice=float(dice),\n        precision=float(precision),\n        recall=float(recall),\n        true_positive=intersection,\n        false_positive=predicted_area - intersection,\n        false_negative=truth_area - intersection,\n    )\n\n\n@dataclass\nclass EvaluationReport:\n    \"\"\"Per-example rows plus the mean of each metric.\n\n    Attributes:\n        rows: one dict per example, containing the metrics, the confidence and\n            the runtime when available.\n        metric_names: the metrics that the mean is computed over.\n    \"\"\"\n\n    rows: List[Dict[str, Any]]\n    metric_names: Tuple[str, ...] = (\"iou\", \"dice\", \"precision\", \"recall\")\n\n    def mean(self, metric: str = \"iou\") -> float:\n        values = [float(row[metric]) for row in self.rows if metric in row]\n        return float(np.mean(values)) if values else float(\"nan\")\n\n    def summary(self) -> Dict[str, float]:\n        return {name: self.mean(name) for name in self.metric_names}\n\n    def to_table(self) -> str:\n        if not self.rows:\n            return \"no evaluated examples\"\n        header = f\"{'example':<24}{'IoU':>8}{'Dice':>8}{'P':>8}{'R':>8}{'conf':>8}{'sec':>8}\"\n        lines = [header, \"-\" * len(header)]\n        for row in self.rows:\n            lines.append(\n                f\"{str(row.get('example', '')):<24}\"\n                f\"{row.get('iou', float('nan')):>8.3f}\"\n                f\"{row.get('dice', float('nan')):>8.3f}\"\n                f\"{row.get('precision', float('nan')):>8.3f}\"\n                f\"{row.get('recall', float('nan')):>8.3f}\"\n                f\"{row.get('confidence', float('nan')):>8.3f}\"\n                f\"{row.get('seconds', float('nan')):>8.2f}\"\n            )\n        lines.append(\"-\" * len(header))\n        summary = self.summary()\n        lines.append(\n            f\"{'mean':<24}\" + \"\".join(f\"{summary[name]:>8.3f}\" for name in self.metric_names)\n        )\n        return \"\\n\".join(lines)\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\"rows\": self.rows, \"summary\": self.summary(), \"num_examples\": len(self.rows)}\n\n\ndef evaluate_results(\n    examples: Sequence[Tuple[str, Optional[MaskInput], MaskInput, Any]],\n) -> EvaluationReport:\n    \"\"\"Score `(name, prediction, ground_truth, result)` tuples into a report.\n\n    The fourth element is the `OneShotResult` the prediction came from, used only\n    to attach the confidence and the runtime; `None` is accepted.\n    \"\"\"\n    rows: List[Dict[str, Any]] = []\n    for name, prediction, ground_truth, result in examples:\n        metrics = evaluate_mask(prediction, ground_truth)\n        row: Dict[str, Any] = {\"example\": name, **metrics.to_dict()}\n        if result is not None:\n            row[\"confidence\"] = float(getattr(result, \"confidence\", float(\"nan\")))\n            row[\"seconds\"] = float(getattr(result, \"runtime\", {}).get(\"total\", float(\"nan\")))\n        rows.append(row)\n    return EvaluationReport(rows=rows)\n\n\nSUCCESS_THRESHOLDS = (0.5, 0.7, 0.9)\n\n\ndef summarize_values(values: Sequence[float],\n                     thresholds: Sequence[float] = SUCCESS_THRESHOLDS) -> Dict[str, float]:\n    \"\"\"Distribution summary for one metric across many samples.\n\n    A mean alone hides too much: a handful of near-perfect cases can lift the\n    average while most pairs fail, and a failure with an empty prediction scores\n    `0.0` rather than disappearing. The median and the success rates at fixed IoU\n    thresholds are what make a run comparable to another run.\n    \"\"\"\n    if not len(values):\n        return {}\n    array = np.asarray([float(value) for value in values], dtype=float)\n    summary: Dict[str, float] = {\n        \"count\": int(array.size),\n        \"mean\": float(array.mean()),\n        \"median\": float(np.median(array)),\n        \"std\": float(array.std(ddof=0)),\n        \"min\": float(array.min()),\n        \"max\": float(array.max()),\n        \"q25\": float(np.percentile(array, 25)),\n        \"q75\": float(np.percentile(array, 75)),\n    }\n    for threshold in thresholds:\n        key = f\"success_rate_at_{threshold:g}\".replace(\".\", \"_\")\n        summary[key] = float((array >= threshold).mean())\n    return summary\n\n\ndef aggregate_rows(rows: Sequence[Dict[str, Any]],\n                   metrics: Sequence[str] = (\"iou\", \"dice\", \"precision\", \"recall\"),\n                   by: Optional[Sequence[str]] = None,\n                   min_group: int = 1) -> Dict[str, Any]:\n    \"\"\"Summarise metric rows overall and per group.\n\n    Args:\n        rows: rows as produced by the benchmark runner, each holding at least one\n            metric column.\n        metrics: metric column names to summarise.\n        by: optional columns to group by, such as `category` or `scale_condition`.\n        min_group: groups with fewer samples than this are pooled into `other`,\n            so a per-category table cannot be read off a single pair. The default\n            keeps every group visible; its `count` says how thin it is.\n\n    Returns:\n        A dict with `overall` metrics, a `groups` mapping, and the counts that\n        produced them.\n    \"\"\"\n    payload: Dict[str, Any] = {\"overall\": {}, \"groups\": {}, \"rows\": len(rows)}\n    for metric in metrics:\n        values = [row[metric] for row in rows if metric in row and row[metric] is not None]\n        payload[\"overall\"][metric] = summarize_values(values)\n\n    for column in (by or ()):\n        buckets: Dict[str, List[Dict[str, Any]]] = {}\n        for row in rows:\n            key = str(row.get(column, \"unknown\"))\n            buckets.setdefault(key, []).append(row)\n        small = sorted(key for key, items in buckets.items() if len(items) < min_group)\n        if small:\n            pooled = [row for key in small for row in buckets.pop(key)]\n            if pooled:\n                buckets[\"other\"] = pooled\n        payload[\"groups\"][column] = {\n            key: {\n                \"count\": len(items),\n                **{metric: summarize_values([item[metric] for item in items\n                                            if metric in item and item[metric] is not None])\n                   for metric in metrics},\n            }\n            for key, items in sorted(buckets.items())\n        }\n    return payload\n\n\ndef iou_histogram(values: Sequence[float], bins: int = 10,\n                  thresholds: Sequence[float] = (0.25, 0.5, 0.75, 0.9)) -> Dict[str, Any]:\n    \"\"\"Histogram of IoU plus the counts needed to draw a precision/recall curve.\n\n    Both come from the same numbers on purpose: the histogram shows the shape of\n    the distribution, and the threshold counts answer \"how many pairs were good\n    enough to be usable\", which is a different question.\n    \"\"\"\n    if not len(values):\n        return {\"counts\": [], \"bin_edges\": [], \"thresholds\": {}}\n    array = np.asarray([float(value) for value in values], dtype=float)\n    counts, edges = np.histogram(np.clip(array, 0.0, 1.0), bins=bins, range=(0.0, 1.0))\n    return {\n        \"counts\": [int(value) for value in counts],\n        \"bin_edges\": [float(value) for value in edges],\n        \"mean\": float(array.mean()),\n        \"median\": float(np.median(array)),\n        \"thresholds\": {f\"{threshold:g}\": int((array >= threshold).sum())\n                       for threshold in thresholds},\n        \"count\": int(array.size),\n    }\n\n",
    "evaluation/runtime.py": "\"\"\"Timing, seeding and provenance for benchmark runs.\n\nPhase 1 reported a runtime per example, which is enough to see that the pipeline\nworks. A benchmark needs more than that, because a mean IoU is meaningless\nwithout knowing which machine produced it and how long each stage took. This\nmodule collects the three things a result depends on: wall-clock time per stage,\nGPU memory when a GPU is present, and a description of the environment that\nproduced the numbers.\n\nNothing here changes how the pipeline computes anything. The stages are measured\naround calls the caller already makes, so timing never alters the algorithm:\n\n    with Stopwatch() as watch:\n        result = pipeline.segment_object(reference_image, reference_mask, target_image)\n    record = watch.to_record()\n\nGPU timings are only meaningful when device work is finished, so every measurement\nsynchronises CUDA first when it is available; without that, `cuda.synchronize`\ntimings drift with queueing rather than with compute.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport os\nimport platform\nimport subprocess\nimport sys\nimport time\nfrom contextlib import contextmanager\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, Iterator, List, Optional, Sequence\n\nEPSILON = 1e-9\n\n\ndef synchronize(device: Any = None) -> None:\n    \"\"\"Wait for queued GPU work so a timing measures compute, not queueing.\"\"\"\n    import torch\n\n    if not torch.cuda.is_available():\n        return\n    if device is None or \"cuda\" in str(device):\n        torch.cuda.synchronize()\n\n\ndef gpu_memory_mb(device: Any = None) -> Optional[float]:\n    \"\"\"Peak CUDA memory in megabytes, or `None` when there is no GPU to measure.\"\"\"\n    try:\n        import torch\n\n        if not torch.cuda.is_available():\n            return None\n        if device is not None and \"cuda\" not in str(device):\n            return None\n        index = 0 if device is None else torch.device(device).index or 0\n        return float(torch.cuda.max_memory_allocated(index)) / (1024 ** 2)\n    except Exception:  # noqa: BLE001 - reporting memory must never fail a run\n        return None\n\n\ndef reset_gpu_memory(device: Any = None) -> None:\n    \"\"\"Clear the peak-memory counter so each sample is measured on its own.\"\"\"\n    try:\n        import torch\n\n        if torch.cuda.is_available() and (device is None or \"cuda\" in str(device)):\n            index = 0 if device is None else torch.device(device).index or 0\n            torch.cuda.reset_peak_memory_stats(index)\n    except Exception:  # noqa: BLE001\n        pass\n\n\n@dataclass\nclass RuntimeRecord:\n    \"\"\"Wall-clock seconds for one unit of work, split by stage.\n\n    Attributes:\n        stages: seconds per named stage, in insertion order.\n        total_seconds: sum of every stage measured.\n        peak_gpu_mb: peak CUDA allocation during the work, if any.\n        device: the device string the work ran on, for the record.\n    \"\"\"\n\n    stages: Dict[str, float] = field(default_factory=dict)\n    peak_gpu_mb: Optional[float] = None\n    device: str = \"cpu\"\n\n    @property\n    def total_seconds(self) -> float:\n        return float(sum(self.stages.values()))\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            \"stages\": {name: float(value) for name, value in self.stages.items()},\n            \"total_seconds\": self.total_seconds,\n            \"peak_gpu_mb\": self.peak_gpu_mb,\n            \"device\": self.device,\n        }\n\n\nclass Stopwatch:\n    \"\"\"Context manager that times the block and attributes it to one stage.\n\n        with Stopwatch() as watch:\n            with watch.stage(\"features\"):\n                features = encoder(image)\n    \"\"\"\n\n    def __init__(self, device: Any = None, reset_memory: bool = False) -> None:\n        self.device = str(device) if device is not None else \"cpu\"\n        self.record = RuntimeRecord(device=self.device)\n        self._started = 0.0\n        self._open = False\n        if reset_memory:\n            reset_gpu_memory(device)\n\n    def __enter__(self) -> \"Stopwatch\":\n        return self\n\n    def __exit__(self, exc_type: Any, exc: Any, traceback: Any) -> bool:\n        # The record is finished by `to_record`, so leaving the block only has to\n        # hand control back: an exception inside a stage must propagate unchanged.\n        return False\n\n    @contextmanager\n    def stage(self, name: str) -> Iterator[None]:\n        \"\"\"Time one named stage and add it to the record.\"\"\"\n        if self._open:\n            raise RuntimeError(\"Stopwatch stages cannot be nested\")\n        synchronize(self.device)\n        started = time.perf_counter()\n        self._open = True\n        try:\n            yield\n        finally:\n            synchronize(self.device)\n            self.record.stages[name] = self.record.stages.get(name, 0.0) + (\n                time.perf_counter() - started\n            )\n            self._open = False\n\n    def to_record(self) -> RuntimeRecord:\n        \"\"\"Finish the record, sampling peak GPU memory once at the end.\"\"\"\n        synchronize(self.device)\n        self.record.peak_gpu_mb = gpu_memory_mb(self.device)\n        return self.record\n\n\ndef set_seed(seed: int) -> int:\n    \"\"\"Seed every source of randomness this project can reach.\n\n    Returns the seed so a caller can record it next to the results.\n    \"\"\"\n    seed = int(seed)\n    os.environ[\"PYTHONHASHSEED\"] = str(seed)\n    import random\n\n    import numpy as np\n\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n    except Exception:  # noqa: BLE001 - torch is optional for pure-CPU bookkeeping\n        pass\n    return seed\n\n\ndef git_revision(path: Optional[str] = None) -> Optional[str]:\n    \"\"\"Current commit hash, or `None` when git is unavailable or this is not a repo.\"\"\"\n    try:\n        result = subprocess.run(\n            [\"git\", \"rev-parse\", \"HEAD\"],\n            cwd=path or os.getcwd(), capture_output=True, text=True, timeout=10, check=False,\n        )\n        revision = result.stdout.strip()\n        return revision or None\n    except (OSError, subprocess.SubprocessError):\n        return None\n\n\ndef package_version(name: str) -> Optional[str]:\n    \"\"\"Version of an installed distribution, or `None` when it is absent.\"\"\"\n    try:\n        module = __import__(name)\n        return str(getattr(module, \"__version__\", \"unknown\"))\n    except Exception:  # noqa: BLE001\n        return None\n\n\ndef describe_environment(device: Any = None) -> Dict[str, Any]:\n    \"\"\"Record everything about the machine that could change a number.\n\n    The point is that a reader can tell whether two result files are comparable\n    without trusting the person who produced them.\n    \"\"\"\n    environment: Dict[str, Any] = {\n        \"python\": sys.version.split()[0],\n        \"platform\": platform.platform(),\n        \"machine\": platform.machine(),\n        \"processor\": platform.processor(),\n        \"packages\": {\n            name: package_version(name)\n            for name in (\"torch\", \"torchvision\", \"transformers\", \"numpy\", \"scipy\",\n                         \"matplotlib\", \"PIL\")\n        },\n        \"git_revision\": git_revision(),\n    }\n    try:\n        import torch\n\n        environment[\"torch_cuda_build\"] = str(torch.version.cuda)\n        environment[\"cuda_available\"] = bool(torch.cuda.is_available())\n        if torch.cuda.is_available():\n            environment[\"gpu_count\"] = int(torch.cuda.device_count())\n            environment[\"gpu_names\"] = [\n                torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())\n            ]\n        environment[\"device\"] = str(device) if device is not None else \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        if hasattr(torch.backends, \"cudnn\"):\n            environment[\"cudnn_version\"] = str(torch.backends.cudnn.version())\n    except Exception as error:  # noqa: BLE001\n        environment[\"torch_error\"] = str(error)\n    try:\n        import transformers\n\n        environment[\"transformers_cache\"] = os.environ.get(\"HF_HOME\", \"\")\n    except Exception:  # noqa: BLE001\n        pass\n    return environment\n\n\ndef describe_models(pipeline: Any) -> Dict[str, Any]:\n    \"\"\"Model identifiers and parameter counts for the frozen encoders.\"\"\"\n    described: Dict[str, Any] = {}\n    for name in (\"encoder\", \"segmenter\"):\n        module = getattr(pipeline, name, None)\n        if module is None:\n            continue\n        model = getattr(module, \"model\", None)\n        parameters = None\n        trainable = None\n        if model is not None and hasattr(model, \"parameters\"):\n            try:\n                parameters = int(sum(p.numel() for p in model.parameters()))\n                trainable = int(sum(p.numel() for p in model.parameters() if p.requires_grad))\n            except Exception:  # noqa: BLE001\n                parameters = None\n        described[name] = {\n            \"model_id\": str(getattr(module, \"model_id\", \"unknown\")),\n            \"parameters\": parameters,\n            \"trainable_parameters\": trainable,\n            \"device\": str(getattr(module, \"device\", \"unknown\")),\n        }\n    for key in (\"candidate_threshold\", \"candidate_min_area\", \"candidate_top_k\",\n                \"max_candidates\", \"blend_alpha\"):\n        if hasattr(pipeline, key):\n            described[key] = getattr(pipeline, key)\n    return described\n\n\ndef build_provenance(pipeline: Any = None, device: Any = None, seed: Optional[int] = None,\n                     dataset: Optional[Dict[str, Any]] = None,\n                     extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:\n    \"\"\"Assemble the provenance block saved beside a set of results.\"\"\"\n    provenance: Dict[str, Any] = {\n        \"environment\": describe_environment(device),\n        \"created_unix\": int(time.time()),\n        \"created_utc\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n    }\n    if seed is not None:\n        provenance[\"seed\"] = int(seed)\n    if pipeline is not None:\n        provenance[\"pipeline\"] = describe_models(pipeline)\n    if dataset is not None:\n        provenance[\"dataset\"] = dataset\n    if extra:\n        provenance.update(extra)\n    return provenance\n\n\ndef write_json(path: str, payload: Any) -> str:\n    \"\"\"Write JSON with sorted keys and a trailing newline, creating parent folders.\"\"\"\n    import json\n\n    directory = os.path.dirname(os.path.abspath(path))\n    if directory:\n        os.makedirs(directory, exist_ok=True)\n    with open(path, \"w\", encoding=\"utf-8\") as handle:\n        json.dump(payload, handle, indent=2, sort_keys=True, default=_json_default)\n        handle.write(\"\\n\")\n    return path\n\n\ndef _json_default(value: Any) -> Any:\n    try:\n        import numpy as np\n\n        if isinstance(value, (np.integer,)):\n            return int(value)\n        if isinstance(value, (np.floating,)):\n            return float(value)\n        if isinstance(value, np.ndarray):\n            return value.tolist()\n    except Exception:  # noqa: BLE001\n        pass\n    return str(value)\n\n\ndef summarize_times(values: Sequence[float]) -> Dict[str, float]:\n    \"\"\"Mean, median, spread and extremes of a list of durations in seconds.\"\"\"\n    import numpy as np\n\n    if not values:\n        return {}\n    array = np.asarray(list(values), dtype=float)\n    return {\n        \"count\": int(array.size),\n        \"mean_seconds\": float(array.mean()),\n        \"median_seconds\": float(np.median(array)),\n        \"std_seconds\": float(array.std(ddof=0)),\n        \"min_seconds\": float(array.min()),\n        \"max_seconds\": float(array.max()),\n        \"p95_seconds\": float(np.percentile(array, 95)),\n    }\n",
    "matching/correspondence.py": "\"\"\"Steps 3 and 4 - target representation and feature correspondence.\n\nStep 3 wraps the target image features in a `TargetRepresentation`: every patch\nof the target keeps its features and its grid position, so a patch-level\ncomparison is a plain matrix product.\n\nStep 4 turns that into a *correspondence field*. For each target patch we\nmeasure how similar it is to the reference object and resample the result to\npixel resolution, producing the `similarity_map` the later stages consume.\n\nTwo aggregations are available:\n\n* `max` - the single best-matching reference patch per target patch. Sharp and\n  good for texture repetition, but sensitive to a few spuriously high scores.\n* `prototype` - similarity to the coverage-weighted mean reference token. More\n  robust, blurrier, and it describes the object as a whole rather than by its\n  most distinctive patch.\n\n`both` returns the two maps separately plus their average, which is the default\nin the pipeline: agreement between the two is a cheap, honest confidence signal.\n\nSimilarity is computed on L2-normalised tokens, so the cosine similarity is a\ndot product. Patch scores are finally min-max normalised using robust percentile\nclips, which keeps a handful of extreme values from flattening the rest of the\nmap.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport warnings\nfrom dataclasses import dataclass\nfrom typing import Any, Dict, Optional, Tuple, Union\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nfrom models.feature_encoder import FeatureMap\nfrom matching.reference_representation import ReferenceRepresentation\n\nAGGREGATIONS = (\"max\", \"prototype\", \"both\")\n\n\n@dataclass\nclass TargetRepresentation:\n    \"\"\"Features of the target image, flattened for patch-wise comparison.\n\n    Attributes:\n        tokens: (M, C) L2-normalised target patch tokens, row-major over the grid.\n        features: the underlying `FeatureMap`, kept for visualisation.\n    \"\"\"\n\n    tokens: torch.Tensor\n    features: FeatureMap\n\n    @property\n    def grid_size(self) -> Tuple[int, int]:\n        return self.features.grid_size\n\n    @property\n    def num_patches(self) -> int:\n        return int(self.tokens.shape[0])\n\n    def positions(self) -> torch.Tensor:\n        \"\"\"(M, 2) normalised (x, y) centres of the target patches.\"\"\"\n        positions = self.features.grid_positions()\n        grid_h, grid_w = self.grid_size\n        return positions.reshape(grid_h * grid_w, 2)\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            \"model_id\": self.features.model_id,\n            \"image_size\": list(self.features.image_size),\n            \"grid_size\": list(self.grid_size),\n            \"num_patches\": self.num_patches,\n            \"channels\": int(self.tokens.shape[1]),\n        }\n\n\ndef build_target_representation(features: FeatureMap) -> TargetRepresentation:\n    \"\"\"Flatten the target feature map into row-major patch rows.\"\"\"\n    grid_h, grid_w, _ = features.tokens_hwc().shape\n    tokens = features.tokens_hwc().reshape(grid_h * grid_w, -1).contiguous()\n    return TargetRepresentation(tokens=tokens, features=features)\n\n\n@dataclass\nclass CorrespondenceResult:\n    \"\"\"Pixel-level agreement between the reference object and the target.\n\n    Attributes:\n        similarity_map: (H, W) float map in [0, 1] at the target's pixel size.\n        patch_scores: (M,) normalised score per target patch.\n        patch_best_match: (M,) index of the best-matching reference patch.\n        prototype_map: (H, W) prototype-only map when `aggregate=\"both\"`.\n        aggregate: which aggregation produced `similarity_map`.\n        reference_centroid_distance: normalised distance between the similarity\n            peak and the location the reference centroid would project to; used\n            as a sanity signal, not as ground truth.\n    \"\"\"\n\n    similarity_map: np.ndarray\n    patch_scores: np.ndarray\n    patch_best_match: np.ndarray\n    grid_size: Tuple[int, int]\n    aggregate: str\n    prototype_map: Optional[np.ndarray] = None\n    reference_centroid_distance: Optional[float] = None\n\n    @property\n    def shape(self) -> Tuple[int, int]:\n        return tuple(self.similarity_map.shape[:2])\n\n    def peak_pixel(self) -> Tuple[int, int]:\n        \"\"\"(x, y) pixel coordinate of the most similar target patch.\"\"\"\n        grid_h, grid_w = self.grid_size\n        index = int(np.argmax(self.patch_scores))\n        row, col = divmod(index, grid_w)\n        height, width = self.similarity_map.shape[:2]\n        scale_x = width / grid_w\n        scale_y = height / grid_h\n        return ((col + 0.5) * scale_x, (row + 0.5) * scale_y)\n\n    def peak_score(self) -> float:\n        return float(self.patch_scores.max())\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            \"aggregate\": self.aggregate,\n            \"peak_score\": self.peak_score(),\n            \"peak_pixel\": list(self.peak_pixel()),\n            \"grid_size\": list(self.grid_size),\n            \"reference_centroid_distance\": self.reference_centroid_distance,\n        }\n\n\ndef align_reference(\n    reference: \"ReferenceRepresentation\", device: Union[str, torch.device]\n) -> \"ReferenceRepresentation\":\n    \"\"\"Return `reference` with every tensor on `device`.\n\n    A representation restored with `ReferenceRepresentation.load` is always on the\n    CPU, because that is where NumPy arrays live. Comparing it with GPU target\n    features then fails deep inside a matmul with a bare \"expected all tensors to\n    be on the same device\", which says nothing about the cause. Reconciling here\n    keeps the cache usable across devices and names the move in a warning.\n\n    Returns `reference` unchanged when it is already on `device`.\n    \"\"\"\n    wanted = torch.device(device)\n    tensors = (reference.tokens, reference.weights, reference.coverage)\n    if all(tensor.device == wanted for tensor in tensors):\n        return reference\n    warnings.warn(\n        f\"moving the reference representation from {reference.tokens.device} to {wanted}; \"\n        \"a cached reference is stored on the CPU, so this is expected on a GPU run\",\n        stacklevel=2,\n    )\n    return reference.to(wanted)\n\n\ndef cosine_similarity(\n    target: TargetRepresentation,\n    reference: ReferenceRepresentation,\n    aggregate: str = \"both\",\n    smooth_sigma: Optional[float] = 1.5,\n    clip_percentile: float = 99.0,\n    min_percentile: float = 1.0,\n) -> CorrespondenceResult:\n    \"\"\"Compare target patches with the reference object patches.\n\n    Args:\n        target: features of the target image.\n        reference: representation of the reference object.\n        aggregate: one of `\"max\"`, `\"prototype\"`, `\"both\"`.\n        smooth_sigma: Gaussian sigma in pixels applied to the similarity map;\n            `None` disables smoothing.\n        clip_percentile: upper percentile used before the min-max normalisation.\n        min_percentile: lower percentile used before the min-max normalisation.\n\n    Returns:\n        The `CorrespondenceResult` with a pixel-resolution similarity map.\n    \"\"\"\n    if aggregate not in AGGREGATIONS:\n        raise ValueError(f\"aggregate must be one of {AGGREGATIONS}, got {aggregate!r}\")\n    if target.tokens.shape[1] != reference.tokens.shape[1]:\n        raise ValueError(\n            \"Reference and target features are not comparable: channel counts differ \"\n            f\"({reference.tokens.shape[1]} vs {target.tokens.shape[1]}). Use the same encoder for both.\"\n        )\n    reference = align_reference(reference, target.tokens.device)\n\n    scores = target.tokens @ reference.tokens.permute(1, 0).contiguous()\n    best_index = scores.argmax(dim=1)\n    max_scores = scores.max(dim=1).values\n    prototype = reference.weighted_mean_token()\n    prototype_scores = target.tokens @ prototype\n\n    image_size = tuple(target.features.image_size)\n    max_map = _to_pixel_map(max_scores, target.features, image_size)\n    prototype_map = _to_pixel_map(prototype_scores, target.features, image_size)\n\n    if aggregate == \"max\":\n        patch_scores, combined = max_scores, max_map\n    elif aggregate == \"prototype\":\n        patch_scores, combined = prototype_scores, prototype_map\n    else:\n        patch_scores = 0.5 * (max_scores + prototype_scores)\n        combined = 0.5 * (_normalise(max_map) + _normalise(prototype_map))\n\n    combined = _smooth(combined, smooth_sigma)\n    combined = _normalise(combined, clip_percentile, min_percentile)\n\n    if aggregate == \"both\":\n        return CorrespondenceResult(\n            similarity_map=combined,\n            patch_scores=_normalise_series(patch_scores, clip_percentile, min_percentile),\n            patch_best_match=best_index.cpu().numpy(),\n            grid_size=target.features.grid_size,\n            aggregate=aggregate,\n            prototype_map=prototype_map,\n            reference_centroid_distance=_centroid_distance(combined, reference.centroid),\n        )\n    return CorrespondenceResult(\n        similarity_map=combined,\n        patch_scores=_normalise_series(patch_scores, clip_percentile, min_percentile),\n        patch_best_match=best_index.cpu().numpy(),\n        grid_size=target.features.grid_size,\n        aggregate=aggregate,\n        prototype_map=prototype_map if aggregate == \"prototype\" else None,\n        reference_centroid_distance=_centroid_distance(combined, reference.centroid),\n    )\n\n\ndef _to_pixel_map(scores: torch.Tensor, features: FeatureMap, image_size: Tuple[int, int]) -> np.ndarray:\n    grid_h, grid_w = features.grid_size\n    grid = scores.reshape(1, 1, grid_h, grid_w)\n    resized = F.interpolate(grid, size=image_size, mode=\"bilinear\", align_corners=False)\n    return resized.squeeze().detach().cpu().numpy().astype(np.float32)\n\n\ndef _normalise(array: np.ndarray, clip_percentile: float = 99.0, min_percentile: float = 1.0) -> np.ndarray:\n    high = float(np.percentile(array, clip_percentile))\n    low = float(np.percentile(array, min_percentile))\n    if high - low < 1e-8:\n        return np.zeros_like(array, dtype=np.float32)\n    return np.clip((array - low) / (high - low), 0.0, 1.0).astype(np.float32)\n\n\ndef _normalise_series(scores: torch.Tensor, clip_percentile: float, min_percentile: float) -> np.ndarray:\n    values = scores.detach().cpu().numpy()\n    return _normalise(values, clip_percentile, min_percentile)\n\n\ndef _smooth(array: np.ndarray, sigma: Optional[float]) -> np.ndarray:\n    if sigma is None or sigma <= 0:\n        return array\n    from scipy.ndimage import gaussian_filter\n\n    return gaussian_filter(array, sigma=sigma, mode=\"nearest\").astype(np.float32)\n\n\ndef _centroid_distance(similarity_map: np.ndarray, reference_centroid: Tuple[float, float]) -> float:\n    height, width = similarity_map.shape[:2]\n    ys, xs = np.nonzero(similarity_map > 0.5)\n    if xs.size == 0:\n        return float(\"nan\")\n    centre_x = float(xs.mean()) / max(width - 1, 1)\n    centre_y = float(ys.mean()) / max(height - 1, 1)\n    return float(np.hypot(centre_x - reference_centroid[0], centre_y - reference_centroid[1]))\n",
    "matching/localization.py": "\"\"\"Step 5 - candidate localisation, independent of segmentation.\n\nThe correspondence map is noisy at the pixel level, so localisation turns it\ninto a small number of ranked candidate regions:\n\n1. threshold the similarity map;\n2. clean the binary map with a morphological closing (and optional opening) to\n   remove speckle and close pinholes;\n3. label the connected components and drop the tiny ones;\n4. turn each component into a `Candidate` carrying a box, a region mask, a\n   score (mean similarity inside the region) and a centroid.\n\nThis stage is deliberately separate from segmentation. It answers *where* the\nobject is and *how much* the features agree there; the segmenter in Step 6\nanswers *which pixels belong to it*. Keeping them apart makes both stages\ninspectable on their own in the notebook, and it means the boxes can be\nevaluated without loading a segmentation model at all.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\n\n@dataclass\nclass Candidate:\n    \"\"\"One ranked region proposed by the correspondence map.\n\n    Attributes:\n        box: (x1, y1, x2, y2) pixel bounding box, integers, `x2`/`y2` exclusive.\n        region: (H, W) boolean mask of the component.\n        score: mean similarity inside the region, in [0, 1].\n        peak: highest similarity inside the region, in [0, 1].\n        area: component area in pixels.\n        area_fraction: share of the image covered by the component.\n        centroid: (x, y) centroid of the component in pixels.\n    \"\"\"\n\n    box: Tuple[int, int, int, int]\n    region: np.ndarray\n    score: float\n    peak: float\n    area: int\n    area_fraction: float\n    centroid: Tuple[float, float]\n\n    @property\n    def width(self) -> int:\n        return self.box[2] - self.box[0]\n\n    @property\n    def height(self) -> int:\n        return self.box[3] - self.box[1]\n\n    @property\n    def center(self) -> Tuple[float, float]:\n        return ((self.box[0] + self.box[2]) / 2.0, (self.box[1] + self.box[3]) / 2.0)\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            \"box\": [int(value) for value in self.box],\n            \"score\": float(self.score),\n            \"peak\": float(self.peak),\n            \"area\": int(self.area),\n            \"area_fraction\": float(self.area_fraction),\n            \"centroid\": [float(value) for value in self.centroid],\n        }\n\n\n@dataclass\nclass LocalizationResult:\n    \"\"\"All candidate regions for one target image, best first.\n\n    Attributes:\n        candidates: ranked `Candidate` list.\n        binary_map: (H, W) boolean thresholded similarity map before filtering.\n        threshold: threshold that produced `binary_map`.\n        similarity_map: the map the threshold was applied to, for debugging.\n    \"\"\"\n\n    candidates: List[Candidate]\n    binary_map: np.ndarray\n    threshold: float\n    similarity_map: Optional[np.ndarray] = field(default=None, repr=False)\n\n    def __len__(self) -> int:\n        return len(self.candidates)\n\n    @property\n    def best(self) -> Optional[Candidate]:\n        return self.candidates[0] if self.candidates else None\n\n    @property\n    def best_region(self) -> Optional[np.ndarray]:\n        best = self.best\n        return None if best is None else best.region\n\n    @property\n    def confidence(self) -> float:\n        return 0.0 if self.best is None else float(self.best.score)\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            \"threshold\": self.threshold,\n            \"num_candidates\": len(self.candidates),\n            \"candidates\": [candidate.to_dict() for candidate in self.candidates],\n        }\n\n\ndef localize(\n    similarity_map: np.ndarray,\n    threshold: float = 0.5,\n    min_area_fraction: float = 0.002,\n    closing: int = 7,\n    opening: int = 0,\n    max_candidates: int = 5,\n) -> LocalizationResult:\n    \"\"\"Extract ranked candidate regions from a similarity map.\n\n    Args:\n        similarity_map: (H, W) map in [0, 1] from Step 4.\n        threshold: similarity above which a pixel belongs to a candidate.\n        min_area_fraction: components smaller than this share of the image are\n            discarded as noise.\n        closing: side of the square structuring element for the morphological\n            closing; `0` disables it.\n        opening: side for the morphological opening; `0` disables it.\n        max_candidates: how many components to keep after ranking.\n\n    Returns:\n        A `LocalizationResult` whose candidates are sorted by descending score.\n    \"\"\"\n    if similarity_map.ndim != 2:\n        raise ValueError(f\"similarity_map must be 2-D, got shape {similarity_map.shape}\")\n    if not 0.0 < threshold < 1.0:\n        raise ValueError(f\"threshold must be in (0, 1), got {threshold}\")\n\n    from scipy.ndimage import binary_closing, binary_opening, label as connected_components\n\n    binary = similarity_map >= threshold\n    cleaned = binary\n    if closing:\n        cleaned = binary_closing(cleaned, structure=np.ones((closing, closing), dtype=bool))\n    if opening:\n        cleaned = binary_opening(cleaned, structure=np.ones((opening, opening), dtype=bool))\n    cleaned = np.asarray(cleaned, dtype=bool)\n\n    labels, count = connected_components(cleaned)\n    height, width = cleaned.shape\n    min_area = max(1, int(round(min_area_fraction * height * width)))\n\n    candidates: List[Candidate] = []\n    for label in range(1, count + 1):\n        region = labels == label\n        if int(region.sum()) < min_area:\n            continue\n        candidates.append(_make_candidate(region, similarity_map))\n\n    candidates.sort(key=lambda candidate: (candidate.score, candidate.peak, candidate.area), reverse=True)\n    return LocalizationResult(\n        candidates=candidates[:max_candidates],\n        binary_map=binary,\n        threshold=threshold,\n        similarity_map=similarity_map,\n    )\n\n\ndef _make_candidate(region: np.ndarray, similarity_map: np.ndarray) -> Candidate:\n    ys, xs = np.nonzero(region)\n    y1, y2 = int(ys.min()), int(ys.max()) + 1\n    x1, x2 = int(xs.min()), int(xs.max()) + 1\n    values = similarity_map[region]\n    area = int(region.sum())\n    total = float(region.size)\n    return Candidate(\n        box=(x1, y1, x2, y2),\n        region=region,\n        score=float(values.mean()),\n        peak=float(values.max()),\n        area=area,\n        area_fraction=area / total,\n        centroid=(float(xs.mean()), float(ys.mean())),\n    )\n\n\ndef merge_boxes(\n    boxes: Sequence[Tuple[int, int, int, int]], iou_threshold: float = 0.5\n) -> List[Tuple[int, int, int, int]]:\n    \"\"\"Greedy non-maximum suppression over boxes, used to combine prompts.\"\"\"\n    remaining = sorted(boxes, key=lambda box: (box[2] - box[0]) * (box[3] - box[1]), reverse=True)\n    kept: List[Tuple[int, int, int, int]] = []\n    for box in remaining:\n        if all(_box_iou(box, other) < iou_threshold for other in kept):\n            kept.append(box)\n    return kept\n\n\ndef _box_iou(first: Tuple[int, int, int, int], second: Tuple[int, int, int, int]) -> float:\n    x1 = max(first[0], second[0])\n    y1 = max(first[1], second[1])\n    x2 = min(first[2], second[2])\n    y2 = min(first[3], second[3])\n    if x2 <= x1 or y2 <= y1:\n        return 0.0\n    intersection = (x2 - x1) * (y2 - y1)\n    first_area = (first[2] - first[0]) * (first[3] - first[1])\n    second_area = (second[2] - second[0]) * (second[3] - second[1])\n    union = first_area + second_area - intersection\n    return intersection / union if union else 0.0\n",
    "matching/reference_representation.py": "\"\"\"Step 2 - the reference object representation.\n\nThe reference image carries a binary mask that says which object we are looking\nfor. This module turns that mask plus the image's dense features into a compact\nbut spatially faithful description of the object:\n\n1. the mask is resized to the exact input grid the encoder saw, so no geometry\n   is guessed;\n2. it is average-pooled by the patch size, giving the fraction of every patch\n   that belongs to the object;\n3. the covered patches become the reference token set, keeping their grid\n   coordinates and their coverage as weights.\n\nSpatial information is preserved twice over: every reference token still knows\nwhere it came from (`grid`), and the full coverage map is stored so callers can\nvisualise or re-threshold the selection. Nothing is collapsed into a single\nvector, because a single vector would destroy the layout that later stages need.\n\nThe representation is a plain dataclass holding torch tensors, so it can be\ncached in memory or written to disk with `save()` / `load()` and reused across\nexperiments.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport json\nimport os\nimport warnings\nfrom dataclasses import dataclass\nfrom typing import Any, Dict, List, Optional, Tuple, Union\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nfrom models.feature_encoder import FeatureMap\n\nMaskInput = Union[np.ndarray, torch.Tensor, \"Any\"]\n\n\ndef to_binary_mask(mask: MaskInput) -> torch.Tensor:\n    \"\"\"Return the mask as a float tensor in [0, 1] shaped (1, 1, H, W).\"\"\"\n    if isinstance(mask, torch.Tensor):\n        tensor = mask.detach().float()\n    elif isinstance(mask, np.ndarray):\n        tensor = torch.from_numpy(np.ascontiguousarray(mask)).float()\n    else:\n        array = np.asarray(mask)\n        if array.dtype != np.float32 and array.dtype != np.float64:\n            array = array.astype(np.float32) / 255.0\n        tensor = torch.from_numpy(np.ascontiguousarray(array)).float()\n    if tensor.ndim == 2:\n        tensor = tensor.unsqueeze(0).unsqueeze(0)\n    elif tensor.ndim == 3:\n        tensor = tensor.unsqueeze(0)\n    if tensor.shape[1] > 1:\n        tensor = tensor[:, :1]\n    return tensor.clamp(0.0, 1.0)\n\n\n@dataclass\nclass ReferenceRepresentation:\n    \"\"\"Description of the reference object, anchored on the patch grid.\n\n    Attributes:\n        tokens: (N, C) L2-normalised features of the object patches.\n        weights: (N,) coverage of each selected patch, sums to the object area\n            expressed in patches.\n        grid: (N, 2) `(row, col)` position of each selected patch.\n        coverage: (h, w) fraction of every patch covered by the reference mask.\n        image_size: (H, W) size of the reference image in pixels.\n        input_size: (H_in, W_in) size the encoder received.\n        model_id: identifier of the encoder that produced the tokens.\n        patch_size: patch side in encoder-input pixels.\n        area_fraction: share of the reference image covered by the mask.\n        centroid: mask centroid as normalised (x, y) in [0, 1].\n        thresholded: whether any patch reached `min_coverage`; `False` means the\n            most-covered patches were used instead and a warning was emitted.\n    \"\"\"\n\n    tokens: torch.Tensor\n    weights: torch.Tensor\n    grid: torch.Tensor\n    coverage: torch.Tensor\n    image_size: Tuple[int, int]\n    input_size: Tuple[int, int]\n    model_id: str\n    patch_size: int\n    area_fraction: float\n    centroid: Tuple[float, float]\n    thresholded: bool = True\n\n    @property\n    def num_patches(self) -> int:\n        return int(self.tokens.shape[0])\n\n    @property\n    def channels(self) -> int:\n        return int(self.tokens.shape[1])\n\n    @property\n    def grid_size(self) -> Tuple[int, int]:\n        return int(self.coverage.shape[0]), int(self.coverage.shape[1])\n\n    def centroid_pixels(self) -> Tuple[float, float]:\n        \"\"\"Centroid in reference-image pixels.\"\"\"\n        return (self.centroid[0] * self.image_size[1], self.centroid[1] * self.image_size[0])\n\n    def weighted_mean_token(self) -> torch.Tensor:\n        \"\"\"(C,) coverage-weighted average of the reference tokens.\"\"\"\n        weights = self.weights.to(device=self.tokens.device, dtype=self.tokens.dtype)\n        weights = weights / weights.sum().clamp_min(1e-8)\n        return F.normalize((self.tokens * weights.unsqueeze(1)).sum(dim=0), dim=0)\n\n    def to(self, device: Union[str, torch.device]) -> \"ReferenceRepresentation\":\n        \"\"\"Copy of this representation with every tensor on `device`.\"\"\"\n        return ReferenceRepresentation(\n            tokens=self.tokens.to(device),\n            weights=self.weights.to(device),\n            grid=self.grid.to(device),\n            coverage=self.coverage.to(device),\n            image_size=self.image_size,\n            input_size=self.input_size,\n            model_id=self.model_id,\n            patch_size=self.patch_size,\n            area_fraction=self.area_fraction,\n            centroid=self.centroid,\n            thresholded=self.thresholded,\n        )\n\n    def coverage_map(self, size: Optional[Tuple[int, int]] = None) -> np.ndarray:\n        \"\"\"Coverage resampled to pixel resolution, as a numpy array.\"\"\"\n        target = tuple(size) if size is not None else self.image_size\n        resized = F.interpolate(\n            self.coverage.unsqueeze(0).unsqueeze(0), size=target, mode=\"bilinear\", align_corners=False\n        )\n        return resized.squeeze().detach().cpu().numpy()\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            \"model_id\": self.model_id,\n            \"patch_size\": self.patch_size,\n            \"image_size\": list(self.image_size),\n            \"input_size\": list(self.input_size),\n            \"grid_size\": list(self.grid_size),\n            \"num_patches\": self.num_patches,\n            \"channels\": self.channels,\n            \"area_fraction\": self.area_fraction,\n            \"centroid\": list(self.centroid),\n            \"thresholded\": self.thresholded,\n            \"coverage\": self.coverage.detach().cpu().numpy().tolist(),\n        }\n\n    def save(self, path: str) -> str:\n        \"\"\"Write the representation to an `.npz` file next to a small `.json`.\"\"\"\n        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)\n        np.savez_compressed(\n            path,\n            tokens=self.tokens.detach().cpu().numpy(),\n            weights=self.weights.detach().cpu().numpy(),\n            grid=self.grid.detach().cpu().numpy(),\n            coverage=self.coverage.detach().cpu().numpy(),\n        )\n        with open(f\"{os.path.splitext(path)[0]}.json\", \"w\", encoding=\"utf-8\") as handle:\n            json.dump(self.to_dict(), handle, indent=2)\n        return os.path.abspath(path)\n\n    @classmethod\n    def load(cls, path: str) -> \"ReferenceRepresentation\":\n        \"\"\"Read back a representation written by `save()`.\"\"\"\n        with np.load(path) as data:\n            tensors = {key: torch.from_numpy(data[key]) for key in data.files}\n        with open(f\"{os.path.splitext(path)[0]}.json\", \"r\", encoding=\"utf-8\") as handle:\n            meta = json.load(handle)\n        return cls(\n            tokens=tensors[\"tokens\"].float(),\n            weights=tensors[\"weights\"].float(),\n            grid=tensors[\"grid\"].long(),\n            coverage=tensors[\"coverage\"].float(),\n            image_size=tuple(meta[\"image_size\"]),\n            input_size=tuple(meta[\"input_size\"]),\n            model_id=meta[\"model_id\"],\n            patch_size=int(meta[\"patch_size\"]),\n            area_fraction=float(meta[\"area_fraction\"]),\n            centroid=tuple(meta[\"centroid\"]),\n            thresholded=bool(meta[\"thresholded\"]),\n        )\n\n\ndef build_reference_representation(\n    features: FeatureMap,\n    reference_mask: MaskInput,\n    min_coverage: float = 0.5,\n    fallback_fraction: float = 0.01,\n) -> ReferenceRepresentation:\n    \"\"\"Select the reference-object patches described by `reference_mask`.\n\n    Args:\n        features: dense features of the reference image.\n        reference_mask: binary mask of the reference object.\n        min_coverage: minimum share of a patch that must be covered for the patch\n            to belong to the object.\n        fallback_fraction: when no patch reaches `min_coverage` (a very small\n            object), the best `fallback_fraction` of patches is used instead.\n\n    Returns:\n        The `ReferenceRepresentation` for this reference.\n    \"\"\"\n    mask = to_binary_mask(reference_mask)\n    if mask.shape[-2:] != tuple(features.image_size):\n        mask = F.interpolate(mask, size=tuple(features.image_size), mode=\"nearest\")\n    if mask.shape[-2:] != tuple(features.input_size):\n        mask = F.interpolate(mask, size=tuple(features.input_size), mode=\"bilinear\", align_corners=False)\n\n    mask = mask.to(device=features.tokens.device)\n    coverage = F.avg_pool2d(mask, kernel_size=features.patch_size, stride=features.patch_size)\n    coverage = coverage.squeeze(0).squeeze(0).clamp(0.0, 1.0)\n\n    area_fraction = float(mask.mean().item())\n\n    ys, xs = torch.nonzero(coverage >= min_coverage, as_tuple=True)\n    thresholded = bool(ys.numel() > 0)\n    if not thresholded:\n        keep = max(1, int(round(fallback_fraction * coverage.numel())))\n        flat = coverage.flatten()\n        top = torch.topk(flat, k=min(keep, flat.numel())).indices\n        ys = torch.div(top, coverage.shape[1], rounding_mode=\"floor\")\n        xs = top % coverage.shape[1]\n        warnings.warn(\n            \"No patch reached min_coverage=\"\n            f\"{min_coverage}; using the {ys.numel()} most covered patches instead. \"\n            \"Lower min_coverage or enlarge the reference object.\",\n            stacklevel=2,\n        )\n\n    grid = torch.stack([ys, xs], dim=1)\n    tokens = features.tokens[:, ys, xs].permute(1, 0).contiguous()\n    weights = coverage[ys, xs].contiguous()\n    if tokens.device != weights.device:  # pragma: no cover - guarded by the mask move above\n        raise RuntimeError(\n            f\"reference tokens are on {tokens.device} but the pooled mask weights are on \"\n            f\"{weights.device}; the mask was not moved onto the feature device\"\n        )\n\n    centroid_y, centroid_x = _mask_centroid(mask)\n    return ReferenceRepresentation(\n        tokens=tokens,\n        weights=weights,\n        grid=grid,\n        coverage=coverage,\n        image_size=tuple(features.image_size),\n        input_size=tuple(features.input_size),\n        model_id=features.model_id,\n        patch_size=features.patch_size,\n        area_fraction=area_fraction,\n        centroid=(centroid_x, centroid_y),\n        thresholded=thresholded,\n    )\n\n\ndef _mask_centroid(mask: torch.Tensor) -> Tuple[float, float]:\n    binary = (mask > 0.5).float()\n    total = binary.sum()\n    if float(total) <= 0.0:\n        return 0.5, 0.5\n    rows = torch.arange(binary.shape[-2], dtype=torch.float32, device=binary.device).view(-1, 1)\n    cols = torch.arange(binary.shape[-1], dtype=torch.float32, device=binary.device).view(1, -1)\n    centre_y = float((binary * rows).sum().item() / total.item())\n    centre_x = float((binary * cols).sum().item() / total.item())\n    height, width = binary.shape[-2], binary.shape[-1]\n    return centre_y / height, centre_x / width\n\n\nclass ReferenceRepresentationStore:\n    \"\"\"In-memory cache of reference representations, keyed by an explicit id.\"\"\"\n\n    def __init__(self) -> None:\n        self._items: Dict[str, ReferenceRepresentation] = {}\n\n    def put(self, key: str, representation: ReferenceRepresentation) -> None:\n        self._items[key] = representation\n\n    def get(self, key: str) -> Optional[ReferenceRepresentation]:\n        return self._items.get(key)\n\n    def keys(self) -> List[str]:\n        return sorted(self._items)\n\n    def clear(self) -> None:\n        self._items.clear()\n\n    def __len__(self) -> int:\n        return len(self._items)\n",
    "models/feature_encoder.py": "\"\"\"Step 1 - spatial visual features from a frozen vision foundation model.\n\nModel choice\n------------\n`facebook/dinov2-base` (DINOv2 ViT-B/14) through `transformers`.\n\n* spatial quality: patch tokens are trained without a text/image alignment loss,\n  so they stay spatially consistent and support patch-level correspondence;\n* availability: the checkpoint is served by the Hugging Face hub and loads with\n  `transformers` alone - no external research repository is cloned or imported;\n* Colab compatibility: ViT-B fits comfortably in a T4 GPU at 518x518 input;\n* compute: ~86M parameters, one forward pass per image, no fine-tuning;\n* suitability for one-shot matching: the same encoder is applied to the\n  reference and the target image, so the two feature grids live in one metric\n  space and cosine similarity is directly meaningful.\n\nOnly the last transformer block is used (`layer=-1`), the standard choice for\nDINOv2 dense prediction.\n\nInterface\n---------\n    encoder = FeatureEncoder(device=\"cuda\")\n    features = encoder.extract(image)          # -> FeatureMap\n    features.tokens                            # (C, h, w) L2-normalised\n    features.grid_positions()                  # (h, w, 2) normalised xy\n\nThe image is resized (never cropped) to a multiple of the patch size, so the\ntoken grid covers the whole image and its geometry stays known exactly. That is\nwhat lets the reference mask be pooled onto the same grid without guessing.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport hashlib\nimport os\nfrom dataclasses import dataclass\nfrom typing import Any, Dict, List, Optional, Sequence, Tuple, Union\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nDEFAULT_ENCODER_ID = \"facebook/dinov2-base\"\nDEFAULT_MAX_SIZE = 518\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\nImageInput = Union[str, \"os.PathLike[str]\", np.ndarray, \"Any\"]\n\n_ENCODER_CACHE: Dict[Tuple[Any, ...], \"FeatureEncoder\"] = {}\n\n\ndef to_pil_image(image: ImageInput):\n    \"\"\"Accept a path, an RGB/RGBA array or a PIL image and return a PIL image.\"\"\"\n    from PIL import Image\n\n    if isinstance(image, (str, os.PathLike)):\n        return Image.open(image).convert(\"RGB\")\n    if isinstance(image, np.ndarray):\n        array = image\n        if array.dtype != np.uint8:\n            array = np.clip(array, 0.0, 1.0) * 255.0\n            array = array.astype(np.uint8)\n        if array.ndim == 2:\n            return Image.fromarray(array, mode=\"L\").convert(\"RGB\")\n        return Image.fromarray(array[..., :3], mode=\"RGB\")\n    return image.convert(\"RGB\")\n\n\n@dataclass\nclass FeatureMap:\n    \"\"\"Dense features for one image, kept on their patch grid.\n\n    Attributes:\n        tokens: (C, h, w) patch tokens, channel-last features folded into a map\n            and L2-normalised when `normalised` is True.\n        image_size: (H, W) size of the source image in pixels.\n        input_size: (H_in, W_in) size the encoder actually received.\n        patch_size: side of one square token in input pixels.\n        model_id: checkpoint identifier, recorded for provenance.\n        layer: transformer block the tokens come from.\n        normalised: whether the tokens are L2-normalised.\n    \"\"\"\n\n    tokens: torch.Tensor\n    image_size: Tuple[int, int]\n    input_size: Tuple[int, int]\n    patch_size: int\n    model_id: str\n    layer: int = -1\n    normalised: bool = True\n\n    @property\n    def grid_size(self) -> Tuple[int, int]:\n        return int(self.tokens.shape[1]), int(self.tokens.shape[2])\n\n    @property\n    def num_patches(self) -> int:\n        return int(self.tokens.shape[1] * self.tokens.shape[2])\n\n    @property\n    def channels(self) -> int:\n        return int(self.tokens.shape[0])\n\n    def to(self, device: Union[str, torch.device]) -> \"FeatureMap\":\n        return FeatureMap(\n            tokens=self.tokens.to(device),\n            image_size=self.image_size,\n            input_size=self.input_size,\n            patch_size=self.patch_size,\n            model_id=self.model_id,\n            layer=self.layer,\n            normalised=self.normalised,\n        )\n\n    def tokens_hwc(self) -> torch.Tensor:\n        \"\"\"(h, w, C) view, the layout used for patch-wise comparisons.\"\"\"\n        return self.tokens.permute(1, 2, 0).contiguous()\n\n    def grid_positions(self, device: Optional[Union[str, torch.device]] = None) -> torch.Tensor:\n        \"\"\"(h, w, 2) centre of every patch as normalised image coordinates in [0, 1].\"\"\"\n        grid_h, grid_w = self.grid_size\n        ys = (torch.arange(grid_h, dtype=torch.float32) + 0.5) / grid_h\n        xs = (torch.arange(grid_w, dtype=torch.float32) + 0.5) / grid_w\n        grid_y, grid_x = torch.meshgrid(ys, xs, indexing=\"ij\")\n        positions = torch.stack([grid_x, grid_y], dim=-1)\n        return positions.to(device or self.tokens.device)\n\n    def pixel_scale(self) -> Tuple[float, float]:\n        \"\"\"Pixels per patch, as (height, width).\"\"\"\n        return (self.input_size[0] / self.grid_size[0], self.input_size[1] / self.grid_size[1])\n\n    def upsample(self, size: Optional[Tuple[int, int]] = None, mode: str = \"bilinear\") -> torch.Tensor:\n        \"\"\"Resample tokens to pixel resolution, returning (H, W, C).\"\"\"\n        target = tuple(size) if size is not None else self.image_size\n        resized = F.interpolate(\n            self.tokens.unsqueeze(0), size=target, mode=mode, align_corners=False\n        )\n        return resized.squeeze(0).permute(1, 2, 0).contiguous()\n\n    def describe(self) -> Dict[str, Any]:\n        return {\n            \"model_id\": self.model_id,\n            \"layer\": self.layer,\n            \"image_size\": list(self.image_size),\n            \"input_size\": list(self.input_size),\n            \"grid_size\": list(self.grid_size),\n            \"num_patches\": self.num_patches,\n            \"channels\": self.channels,\n            \"patch_size\": self.patch_size,\n            \"normalised\": self.normalised,\n        }\n\n\nclass FeatureEncoder:\n    \"\"\"Frozen dense feature extractor.\n\n    Args:\n        model_id: Hugging Face identifier of the vision backbone.\n        device: torch device; `None` picks CUDA when available.\n        dtype: dtype used for inference.\n        max_size: side length (in patches' input pixels) the longest image side is\n            resized to when `image_size` is None.\n        image_size: fixed square input; `None` keeps the aspect ratio and rounds\n            both sides to a multiple of the patch size.\n        layer: transformer block to read (`-1` = last).\n        normalise: L2-normalise tokens so cosine similarity is a dot product.\n        cache_enabled: keep features of recent images in memory.\n        cache_size: number of entries kept.\n    \"\"\"\n\n    def __init__(\n        self,\n        model_id: str = DEFAULT_ENCODER_ID,\n        device: Optional[Union[str, torch.device]] = None,\n        dtype: torch.dtype = torch.float32,\n        max_size: int = DEFAULT_MAX_SIZE,\n        image_size: Optional[int] = None,\n        layer: int = -1,\n        normalise: bool = True,\n        cache_enabled: bool = True,\n        cache_size: int = 8,\n        cache_dir: Optional[str] = None,\n    ) -> None:\n        self.model_id = model_id\n        self.device = torch.device(device) if device is not None else _default_device()\n        self.dtype = dtype\n        self.max_size = int(max_size)\n        self.image_size = image_size\n        self.layer = int(layer)\n        self.normalise = bool(normalise)\n        self.cache_enabled = bool(cache_enabled)\n        self.cache_size = int(cache_size)\n        self.cache_dir = cache_dir\n        self._model: Optional[torch.nn.Module] = None\n        self._cache: \"Dict[Tuple[Any, ...], FeatureMap]\" = {}\n        self._cache_order: List[Tuple[Any, ...]] = []\n\n    def load(self) -> \"FeatureEncoder\":\n        \"\"\"Download (or read from `cache_dir`) and load the frozen backbone.\"\"\"\n        if self._model is not None:\n            return self\n        from transformers import AutoModel\n\n        kwargs: Dict[str, Any] = {}\n        if self.cache_dir:\n            os.makedirs(self.cache_dir, exist_ok=True)\n            kwargs[\"cache_dir\"] = self.cache_dir\n        model = AutoModel.from_pretrained(self.model_id, **kwargs)\n        model.eval().to(device=self.device, dtype=self.dtype)\n        for parameter in model.parameters():\n            parameter.requires_grad_(False)\n        self._model = model\n        return self\n\n    def unload(self) -> None:\n        self._model = None\n        self._cache.clear()\n        self._cache_order.clear()\n        if self.device.type == \"cuda\":\n            torch.cuda.empty_cache()\n\n    @property\n    def is_loaded(self) -> bool:\n        return self._model is not None\n\n    @property\n    def patch_size(self) -> int:\n        self.load()\n        return int(self._model.config.patch_size)\n\n    def describe(self) -> Dict[str, Any]:\n        return {\n            \"model_id\": self.model_id,\n            \"device\": str(self.device),\n            \"dtype\": str(self.dtype),\n            \"layer\": self.layer,\n            \"max_size\": self.max_size,\n            \"image_size\": self.image_size,\n            \"patch_size\": self.patch_size if self.is_loaded else None,\n            \"loaded\": self.is_loaded,\n        }\n\n    def extract(self, image: ImageInput) -> FeatureMap:\n        \"\"\"Return the dense feature map of one image.\"\"\"\n        return self.extract_batch([image])[0]\n\n    def extract_batch(self, images: Sequence[ImageInput]) -> List[FeatureMap]:\n        \"\"\"Extract several images; identical inputs hit the in-memory cache.\n\n        Images that ended up with the same input size are stacked into one\n        forward pass, so a batch of equally shaped images stays efficient while\n        mixed aspect ratios remain correct.\n        \"\"\"\n        pil_images = [to_pil_image(image) for image in images]\n        results: List[Optional[FeatureMap]] = [None] * len(pil_images)\n\n        pending: List[int] = []\n        keys: Dict[int, Tuple[Any, ...]] = {}\n        for index, pil_image in enumerate(pil_images):\n            key = self._cache_key(pil_image)\n            keys[index] = key\n            cached = self._cache.get(key) if self.cache_enabled else None\n            if cached is not None:\n                results[index] = cached\n            else:\n                pending.append(index)\n\n        if pending:\n            self.load()\n            groups: Dict[Tuple[int, int], List[int]] = {}\n            for index in pending:\n                groups.setdefault(self._target_size(pil_images[index].height, pil_images[index].width),\n                                  []).append(index)\n            for input_size, indices in groups.items():\n                batch = torch.stack([self._to_tensor(pil_images[index]) for index in indices])\n                with torch.inference_mode():\n                    output = self._model(\n                        pixel_values=batch.to(device=self.device, dtype=self.dtype),\n                        output_hidden_states=self.layer != -1,\n                    )\n                hidden = output.last_hidden_state if self.layer == -1 else output.hidden_states[self.layer]\n                for slot, index in enumerate(indices):\n                    features = self._to_feature_map(hidden[slot], pil_images[index], input_size)\n                    results[index] = features\n                    if self.cache_enabled:\n                        self._store(keys[index], features)\n\n        return [result for result in results if result is not None]\n\n    def _to_feature_map(self, hidden: torch.Tensor, image, input_size: Tuple[int, int]) -> FeatureMap:\n        grid_h = int(input_size[0]) // self.patch_size\n        grid_w = int(input_size[1]) // self.patch_size\n        row = hidden.reshape(-1, hidden.shape[-1]).float()\n        num_patches = grid_h * grid_w\n        if row.shape[0] < num_patches:\n            raise RuntimeError(\n                f\"{self.model_id} returned {row.shape[0]} tokens, fewer than the {num_patches} \"\n                f\"patches implied by an input of {input_size} with patch size {self.patch_size}\"\n            )\n        patch_tokens = row[-num_patches:]\n        tokens = patch_tokens.reshape(grid_h, grid_w, -1).permute(2, 0, 1).contiguous()\n        if self.normalise:\n            tokens = F.normalize(tokens, dim=0)\n        return FeatureMap(\n            tokens=tokens,\n            image_size=(image.height, image.width),\n            input_size=(int(input_size[0]), int(input_size[1])),\n            patch_size=self.patch_size,\n            model_id=self.model_id,\n            layer=self.layer,\n            normalised=self.normalise,\n        )\n\n    def _to_tensor(self, image) -> torch.Tensor:\n        return self._preprocess(image)\n\n    def _preprocess(self, image) -> torch.Tensor:\n        import torchvision.transforms.functional as TF\n\n        width, height = image.width, image.height\n        target = self._target_size(height, width)\n        resized = TF.resize(image, list(target), interpolation=TF.InterpolationMode.BICUBIC, antialias=True)\n        array = np.asarray(resized, dtype=np.float32) / 255.0\n        tensor = torch.from_numpy(array).permute(2, 0, 1)\n        mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)\n        std = torch.tensor(IMAGENET_STD).view(3, 1, 1)\n        return (tensor - mean) / std\n\n    def _target_size(self, height: int, width: int) -> Tuple[int, int]:\n        patch = self.patch_size\n        if self.image_size:\n            if int(self.image_size) % patch:\n                raise ValueError(f\"image_size={self.image_size} must be a multiple of patch_size={patch}\")\n            return int(self.image_size), int(self.image_size)\n        scale = self.max_size / max(height, width)\n        new_height = max(patch, int(round(height * scale / patch)) * patch)\n        new_width = max(patch, int(round(width * scale / patch)) * patch)\n        return new_height, new_width\n\n    def _cache_key(self, image) -> Tuple[Any, ...]:\n        digest = hashlib.sha1(np.asarray(image).tobytes()).hexdigest()\n        return (self.model_id, self.layer, self.max_size, self.image_size, digest)\n\n    def _store(self, key: Tuple[Any, ...], features: FeatureMap) -> None:\n        self._cache[key] = features\n        self._cache_order.append(key)\n        while len(self._cache_order) > self.cache_size:\n            self._cache.pop(self._cache_order.pop(0), None)\n\n\ndef _default_device() -> torch.device:\n    return torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n\ndef get_default_encoder(device: Optional[Union[str, torch.device]] = None, **kwargs: Any) -> FeatureEncoder:\n    \"\"\"Process-wide encoder, reused across calls so the weights load once.\"\"\"\n    device = torch.device(device) if device is not None else _default_device()\n    key = (str(device), DEFAULT_ENCODER_ID, kwargs.get(\"image_size\"), kwargs.get(\"max_size\"))\n    encoder = _ENCODER_CACHE.get(key)\n    if encoder is None:\n        encoder = FeatureEncoder(device=device, **kwargs)\n        _ENCODER_CACHE[key] = encoder\n    return encoder\n",
    "models/segmenter.py": "\"\"\"Step 6 - promptable segmentation.\n\nModel choice\n------------\n`facebook/sam-vit-base` (Segment Anything ViT-B) through `transformers`.\n\n* it is promptable: a box, a point, or both are exactly the weak supervision a\n  one-shot setting can afford - the pipeline only has a rough region, not a\n  mask;\n* no training, no LoRA, no per-object optimisation, so the whole method stays a\n  preprocessing pipeline plus two frozen forward passes;\n* the hub checkpoint loads with `transformers` alone, so no external research\n  repository is cloned or imported;\n* ViT-B is the smallest SAM variant and runs in about a second per prompt on a\n  T4, which matters because Phase 1 evaluates a handful of examples.\n\nSAM refines the box from localisation; it never sees the reference mask, so the\ntwo stages remain independent and each can be inspected on its own.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport os\nfrom dataclasses import dataclass\nfrom typing import Any, Dict, Optional, Tuple, Union\n\nimport numpy as np\nimport torch\n\nfrom models.feature_encoder import ImageInput, to_pil_image\n\nDEFAULT_SEGMENTER_ID = \"facebook/sam-vit-base\"\nPoint = Union[Tuple[float, float], Tuple[float, float, int]]\n\n\n@dataclass\nclass SegmentationResult:\n    \"\"\"Output of one prompted segmentation call.\n\n    Attributes:\n        mask: (H, W) boolean mask at the original image resolution.\n        score: predicted IoU of the selected mask, as reported by the model.\n        scores: predicted IoU of every returned candidate mask.\n        prompt: the prompt actually sent to the model, for provenance.\n        multimask_output: whether several candidates were requested.\n    \"\"\"\n\n    mask: np.ndarray\n    score: float\n    scores: Tuple[float, ...]\n    prompt: Dict[str, Any]\n    multimask_output: bool = True\n\n    @property\n    def area(self) -> int:\n        return int(self.mask.sum())\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            \"score\": float(self.score),\n            \"scores\": [float(value) for value in self.scores],\n            \"area\": self.area,\n            \"prompt\": self.prompt,\n            \"multimask_output\": self.multimask_output,\n        }\n\n\nclass Segmenter:\n    \"\"\"Frozen SAM wrapper.\n\n    Args:\n        model_id: Hugging Face identifier of the SAM checkpoint.\n        device: torch device; `None` picks CUDA when available.\n        dtype: inference dtype.\n        cache_dir: optional directory for the downloaded weights.\n    \"\"\"\n\n    def __init__(\n        self,\n        model_id: str = DEFAULT_SEGMENTER_ID,\n        device: Optional[Union[str, torch.device]] = None,\n        dtype: torch.dtype = torch.float32,\n        cache_dir: Optional[str] = None,\n    ) -> None:\n        self.model_id = model_id\n        self.device = torch.device(device) if device is not None else _default_device()\n        self.dtype = dtype\n        self.cache_dir = cache_dir\n        self._model: Optional[torch.nn.Module] = None\n        self._processor: Optional[Any] = None\n\n    def load(self) -> \"Segmenter\":\n        \"\"\"Load the checkpoint and its processor, reusing them if already loaded.\"\"\"\n        if self._model is not None:\n            return self\n        from transformers import SamModel, SamProcessor\n\n        kwargs: Dict[str, Any] = {}\n        if self.cache_dir:\n            os.makedirs(self.cache_dir, exist_ok=True)\n            kwargs[\"cache_dir\"] = self.cache_dir\n        self._processor = SamProcessor.from_pretrained(self.model_id, **kwargs)\n        self._model = SamModel.from_pretrained(self.model_id, **kwargs)\n        self._model.eval().to(device=self.device, dtype=self.dtype)\n        for parameter in self._model.parameters():\n            parameter.requires_grad_(False)\n        return self\n\n    def unload(self) -> None:\n        self._model = None\n        self._processor = None\n        if self.device.type == \"cuda\":\n            torch.cuda.empty_cache()\n\n    @property\n    def is_loaded(self) -> bool:\n        return self._model is not None\n\n    def describe(self) -> Dict[str, Any]:\n        return {\n            \"model_id\": self.model_id,\n            \"device\": str(self.device),\n            \"dtype\": str(self.dtype),\n            \"loaded\": self.is_loaded,\n        }\n\n    def segment(\n        self,\n        image: ImageInput,\n        point: Optional[Point] = None,\n        box: Optional[Tuple[float, float, float, float]] = None,\n        multimask_output: bool = True,\n    ) -> SegmentationResult:\n        \"\"\"Segment one object given a point and/or a box.\n\n        Args:\n            image: target image as a path, array or PIL image.\n            point: (x, y) in pixels, optionally (x, y, label) with label 1 for\n                foreground and 0 for background.\n            box: (x1, y1, x2, y2) in pixels.\n            multimask_output: ask SAM for three candidate masks (part, sub-part,\n                whole) and keep the highest scoring one.\n\n        Returns:\n            The `SegmentationResult` at the original image resolution.\n        \"\"\"\n        if point is None and box is None:\n            raise ValueError(\"segment() needs at least one prompt: a point or a box\")\n        self.load()\n        pil_image = to_pil_image(image)\n        model_inputs, meta = self._build_inputs(pil_image, point, box)\n\n        with torch.inference_mode():\n            outputs = self._model(**model_inputs, multimask_output=multimask_output)\n        masks = self._processor.image_processor.post_process_masks(\n            outputs.pred_masks.detach().cpu(),\n            meta[\"original_sizes\"],\n            meta[\"reshaped_input_sizes\"],\n        )\n        scores = outputs.iou_scores.detach().cpu().numpy().reshape(-1)\n        selected = int(np.argmax(scores))\n        masks_for_image = np.asarray(masks[0])\n        candidates = masks_for_image.reshape(-1, *masks_for_image.shape[-2:])\n        if candidates.shape[0] != scores.shape[0]:\n            raise RuntimeError(\n                f\"SAM returned {candidates.shape[0]} masks and {scores.shape[0]} scores for one image; \"\n                \"the model output layout changed and `selected` would be ambiguous\"\n            )\n        return SegmentationResult(\n            mask=np.asarray(candidates[selected], dtype=bool),\n            score=float(scores[selected]),\n            scores=tuple(float(value) for value in scores),\n            prompt=self._prompt_record(point, box),\n            multimask_output=multimask_output,\n        )\n\n    def segment_region(\n        self,\n        image: ImageInput,\n        region: np.ndarray,\n        add_center_point: bool = True,\n        multimask_output: bool = True,\n    ) -> SegmentationResult:\n        \"\"\"Segment from a binary region: its box, and optionally its centre point.\n\n        This is the bridge from Step 5 to Step 6. The box alone can settle on a\n        neighbouring object, so the centre of the region is sent as a foreground\n        point as well.\n        \"\"\"\n        box = region_bbox(region)\n        point: Optional[Point] = None\n        if add_center_point and box is not None:\n            point = ((box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0, 1)\n        return self.segment(image, point=point, box=box, multimask_output=multimask_output)\n\n    def _build_inputs(\n        self, image, point: Optional[Point], box: Optional[Tuple[float, float, float, float]]\n    ) -> Tuple[Dict[str, Any], Dict[str, Any]]:\n        \"\"\"Return `(model_inputs, post_processing_metadata)`.\n\n        The two are kept apart on purpose: `original_sizes` and\n        `reshaped_input_sizes` are processor bookkeeping and must never be\n        forwarded to the model.\n        \"\"\"\n        kwargs: Dict[str, Any] = {}\n        if point is not None:\n            kwargs[\"input_points\"] = [[[float(point[0]), float(point[1])]]]\n            if len(point) > 2:\n                kwargs[\"input_labels\"] = [[[int(point[2])]]]\n        if box is not None:\n            kwargs[\"input_boxes\"] = [[[float(value) for value in box]]]\n        processed = self._processor(image, return_tensors=\"pt\", **kwargs)\n\n        model_inputs: Dict[str, Any] = {\n            \"pixel_values\": processed[\"pixel_values\"].to(device=self.device, dtype=self.dtype),\n        }\n        for key in (\"input_points\", \"input_labels\", \"input_boxes\"):\n            if processed.get(key) is not None:\n                model_inputs[key] = processed[key].to(self.device)\n        meta = {\n            \"original_sizes\": processed[\"original_sizes\"],\n            \"reshaped_input_sizes\": processed[\"reshaped_input_sizes\"],\n        }\n        return model_inputs, meta\n\n    def _prompt_record(\n        self, point: Optional[Point], box: Optional[Tuple[float, float, float, float]]\n    ) -> Dict[str, Any]:\n        record: Dict[str, Any] = {}\n        if point is not None:\n            record[\"point\"] = [float(value) for value in point]\n        if box is not None:\n            record[\"box\"] = [float(value) for value in box]\n        return record\n\n\ndef region_bbox(region: np.ndarray) -> Optional[Tuple[int, int, int, int]]:\n    \"\"\"Bounding box (x1, y1, x2, y2) of a boolean region, or `None` if empty.\"\"\"\n    ys, xs = np.nonzero(np.asarray(region, dtype=bool))\n    if xs.size == 0:\n        return None\n    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1\n\n\ndef _default_device() -> torch.device:\n    return torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n",
    "pipeline/one_shot_segmentation.py": "\"\"\"Step 7 - the one-shot pipeline that ties the stages together.\n\n    result = segment_object(reference_image, reference_mask, target_image)\n    result.predicted_mask      # (H, W) bool\n    result.similarity_map      # (H, W) float in [0, 1]\n    result.candidate_region    # (H, W) bool or None\n    result.confidence          # float\n    result.runtime             # per-stage seconds plus the total\n\nThe data flow is strictly linear, and each stage only depends on the output of\nthe previous one:\n\n    reference image + mask  ->  feature encoder  ->  reference representation\n    target image             ->  feature encoder  ->  target representation\n    the two representations ->  correspondence   ->  similarity map\n    similarity map           ->  localisation     ->  candidate regions\n    best candidate           ->  segmenter (SAM)  ->  predicted mask\n\nNothing is trained, nothing is fitted per object, and the reference\nrepresentation is the only object-specific state, so the same weights serve any\nquery. Every stage can be replaced through the constructor, which is how the\ntest suite exercises the pipeline without downloading checkpoints.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport time\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, List, Optional, Tuple, Union\n\nimport numpy as np\n\nfrom matching.correspondence import (\n    CorrespondenceResult,\n    TargetRepresentation,\n    build_target_representation,\n    cosine_similarity,\n)\nfrom matching.localization import LocalizationResult, localize\nfrom matching.reference_representation import (\n    MaskInput,\n    ReferenceRepresentation,\n    build_reference_representation,\n)\nfrom models.feature_encoder import FeatureEncoder, ImageInput, get_default_encoder\nfrom models.segmenter import Segmenter, SegmentationResult\n\nSTAGES = (\"features\", \"reference\", \"correspondence\", \"localization\", \"segmentation\")\n\n\n@dataclass\nclass OneShotResult:\n    \"\"\"Everything one run produced, with the two artifacts required downstream.\n\n    Attributes:\n        predicted_mask: (H, W) boolean mask of the reference object in the target\n            image, at the target's own resolution. `None` when localisation found\n            nothing, so the failure is visible instead of silently empty.\n        similarity_map: (H, W) float map in [0, 1].\n        candidate_region: (H, W) boolean region of the selected candidate, or\n            `None` when there was none.\n        confidence: agreement of the selected region with the reference object,\n            in [0, 1]; `0.0` when no candidate was found.\n        runtime: seconds spent in each stage plus `total`, and the encoder/segmenter\n            devices for provenance.\n        candidate: the selected `Candidate` (box, score, area, centroid).\n        candidates: every candidate that survived localisation, best first.\n        segmentation: the raw `SegmentationResult` from the segmenter.\n        correspondence: the raw `CorrespondenceResult`.\n        localization: the raw `LocalizationResult`.\n        reference: the reference representation, so a run can be inspected or its\n            features reused.\n    \"\"\"\n\n    predicted_mask: Optional[np.ndarray]\n    similarity_map: np.ndarray\n    candidate_region: Optional[np.ndarray]\n    confidence: float\n    runtime: Dict[str, Any]\n    candidate: Optional[Any] = None\n    candidates: Tuple[Any, ...] = ()\n    segmentation: Optional[SegmentationResult] = None\n    correspondence: Optional[CorrespondenceResult] = field(default=None, repr=False)\n    localization: Optional[LocalizationResult] = field(default=None, repr=False)\n    reference: Optional[ReferenceRepresentation] = field(default=None, repr=False)\n\n    @property\n    def found(self) -> bool:\n        return self.predicted_mask is not None and bool(self.predicted_mask.any())\n\n    @property\n    def mask_area_fraction(self) -> float:\n        if self.predicted_mask is None or self.predicted_mask.size == 0:\n            return 0.0\n        return float(self.predicted_mask.mean())\n\n    def to_dict(self, include_arrays: bool = False) -> Dict[str, Any]:\n        payload: Dict[str, Any] = {\n            \"confidence\": float(self.confidence),\n            \"found\": self.found,\n            \"mask_area_fraction\": self.mask_area_fraction,\n            \"runtime\": self.runtime,\n            \"candidate\": None if self.candidate is None else self.candidate.to_dict(),\n            \"candidates\": [candidate.to_dict() for candidate in self.candidates],\n            \"segmentation\": None if self.segmentation is None else self.segmentation.to_dict(),\n            \"correspondence\": None if self.correspondence is None else self.correspondence.to_dict(),\n            \"localization\": None if self.localization is None else self.localization.to_dict(),\n            \"reference\": None if self.reference is None else {\n                key: value for key, value in self.reference.to_dict().items() if key != \"coverage\"\n            },\n        }\n        if include_arrays:\n            payload[\"predicted_mask\"] = self.predicted_mask\n            payload[\"similarity_map\"] = self.similarity_map\n            payload[\"candidate_region\"] = self.candidate_region\n        return payload\n\n    def save(self, directory: str, stem: str = \"one_shot_result\") -> Dict[str, str]:\n        \"\"\"Write the mask, the similarity map and a JSON summary to `directory`.\"\"\"\n        import json\n        import os\n\n        os.makedirs(directory, exist_ok=True)\n        paths: Dict[str, str] = {}\n\n        if self.predicted_mask is not None:\n            mask_path = os.path.join(directory, f\"{stem}_mask.png\")\n            _write_mask(mask_path, self.predicted_mask)\n            paths[\"mask\"] = mask_path\n\n        map_path = os.path.join(directory, f\"{stem}_similarity.npy\")\n        np.save(map_path, self.similarity_map.astype(np.float32))\n        paths[\"similarity_map\"] = map_path\n\n        json_path = os.path.join(directory, f\"{stem}_summary.json\")\n        with open(json_path, \"w\", encoding=\"utf-8\") as handle:\n            json.dump(self.to_dict(), handle, indent=2, default=float)\n        paths[\"summary\"] = json_path\n        return paths\n\n\nclass OneShotPipeline:\n    \"\"\"Stage-agnostic facade around the seven steps.\n\n    Args:\n        encoder: a `FeatureEncoder`; built from `encoder_kwargs` if omitted.\n        segmenter: a `Segmenter`; built from `segmenter_kwargs` if omitted.\n        device: device passed to whichever model is created here.\n        encoder_kwargs / segmenter_kwargs: forwarded to the two constructors.\n        min_coverage: patch coverage needed for a reference patch to count.\n        aggregate: correspondence aggregation, see `matching.correspondence`.\n        localization_kwargs: defaults for Step 5, overridable per call.\n        segmentation_kwargs: defaults for Step 6, overridable per call.\n    \"\"\"\n\n    def __init__(\n        self,\n        encoder: Optional[FeatureEncoder] = None,\n        segmenter: Optional[Segmenter] = None,\n        device: Optional[Union[str, Any]] = None,\n        encoder_kwargs: Optional[Dict[str, Any]] = None,\n        segmenter_kwargs: Optional[Dict[str, Any]] = None,\n        min_coverage: float = 0.5,\n        aggregate: str = \"both\",\n        localization_kwargs: Optional[Dict[str, Any]] = None,\n        segmentation_kwargs: Optional[Dict[str, Any]] = None,\n    ) -> None:\n        self.encoder = encoder or FeatureEncoder(device=device, **(encoder_kwargs or {}))\n        self.segmenter = segmenter or Segmenter(device=device, **(segmenter_kwargs or {}))\n        self.min_coverage = min_coverage\n        self.aggregate = aggregate\n        self.localization_defaults = dict(localization_kwargs or {})\n        self.segmentation_defaults = dict(segmentation_kwargs or {})\n\n    def run(\n        self,\n        reference_image: ImageInput,\n        reference_mask: MaskInput,\n        target_image: ImageInput,\n        localization_kwargs: Optional[Dict[str, Any]] = None,\n        segmentation_kwargs: Optional[Dict[str, Any]] = None,\n    ) -> OneShotResult:\n        \"\"\"Run the full pipeline for one reference/target pair.\"\"\"\n        timings: Dict[str, float] = {}\n\n        started = time.perf_counter()\n        reference_features, target_features = self._extract(reference_image, target_image)\n        timings[\"features\"] = time.perf_counter() - started\n\n        started = time.perf_counter()\n        reference = build_reference_representation(\n            reference_features, reference_mask, min_coverage=self.min_coverage\n        )\n        target = build_target_representation(target_features)\n        timings[\"reference\"] = time.perf_counter() - started\n\n        started = time.perf_counter()\n        correspondence = cosine_similarity(target, reference, aggregate=self.aggregate)\n        timings[\"correspondence\"] = time.perf_counter() - started\n\n        started = time.perf_counter()\n        localization = localize(similarity_map=correspondence.similarity_map, **{\n            **self.localization_defaults, **(localization_kwargs or {})\n        })\n        timings[\"localization\"] = time.perf_counter() - started\n\n        started = time.perf_counter()\n        segmentation, mask = self._segment(target_image, localization)\n        timings[\"segmentation\"] = time.perf_counter() - started\n\n        best = localization.best\n        runtime: Dict[str, Any] = dict(timings)\n        runtime[\"total\"] = float(sum(timings.values()))\n        runtime[\"encoder_device\"] = str(getattr(self.encoder, \"device\", \"unknown\"))\n        runtime[\"segmenter_device\"] = str(getattr(self.segmenter, \"device\", \"unknown\"))\n        runtime[\"encoder_model\"] = getattr(self.encoder, \"model_id\", \"custom\")\n        runtime[\"segmenter_model\"] = getattr(self.segmenter, \"model_id\", \"custom\")\n\n        return OneShotResult(\n            predicted_mask=mask,\n            similarity_map=correspondence.similarity_map,\n            candidate_region=None if best is None else best.region,\n            confidence=localization.confidence,\n            runtime=runtime,\n            candidate=best,\n            candidates=tuple(localization.candidates),\n            segmentation=segmentation,\n            correspondence=correspondence,\n            localization=localization,\n            reference=reference,\n        )\n\n    def segment_object(\n        self,\n        reference_image: ImageInput,\n        reference_mask: MaskInput,\n        target_image: ImageInput,\n        **kwargs: Any,\n    ) -> OneShotResult:\n        \"\"\"Documented alias of `run`, the public entry point of the project.\"\"\"\n        return self.run(reference_image, reference_mask, target_image, **kwargs)\n\n    def describe(self) -> Dict[str, Any]:\n        return {\n            \"encoder\": self.encoder.describe(),\n            \"segmenter\": self.segmenter.describe(),\n            \"min_coverage\": self.min_coverage,\n            \"aggregate\": self.aggregate,\n            \"localization\": self.localization_defaults,\n            \"segmentation\": self.segmentation_defaults,\n        }\n\n    def _extract(self, reference_image: ImageInput, target_image: ImageInput):\n        reference, target = self.encoder.extract_batch([reference_image, target_image])\n        return reference, target\n\n    def _segment(\n        self, target_image: ImageInput, localization: LocalizationResult\n    ) -> Tuple[Optional[SegmentationResult], Optional[np.ndarray]]:\n        best = localization.best\n        if best is None:\n            return None, None\n        options = {**self.segmentation_defaults, \"region\": best.region}\n        segmentation = self.segmenter.segment_region(target_image, **options)\n        return segmentation, segmentation.mask\n\n\n_DEFAULT_PIPELINE: Optional[OneShotPipeline] = None\n\n\ndef get_default_pipeline(device: Optional[Union[str, Any]] = None, **kwargs: Any) -> OneShotPipeline:\n    \"\"\"Process-wide pipeline so the two checkpoints load only once.\"\"\"\n    global _DEFAULT_PIPELINE\n    if _DEFAULT_PIPELINE is None:\n        _DEFAULT_PIPELINE = OneShotPipeline(device=device, **kwargs)\n    return _DEFAULT_PIPELINE\n\n\ndef segment_object(\n    reference_image: ImageInput,\n    reference_mask: MaskInput,\n    target_image: ImageInput,\n    device: Optional[Union[str, Any]] = None,\n    pipeline: Optional[OneShotPipeline] = None,\n    **kwargs: Any,\n) -> OneShotResult:\n    \"\"\"Segment `target_image` at the object described by `reference_image`/`reference_mask`.\n\n    Args:\n        reference_image: image containing the query object.\n        reference_mask: binary mask of that object in the reference image.\n        target_image: image to segment.\n        device: device for the models when a fresh pipeline is created.\n        pipeline: reuse an existing pipeline instead of the default one.\n        **kwargs: forwarded to `OneShotPipeline.run`.\n\n    Returns:\n        A `OneShotResult` with `predicted_mask`, `candidate_region`,\n        `similarity_map`, `confidence` and `runtime`.\n    \"\"\"\n    active = pipeline or get_default_pipeline(device=device)\n    return active.segment_object(reference_image, reference_mask, target_image, **kwargs)\n\n\ndef _write_mask(path: str, mask: np.ndarray) -> None:\n    from PIL import Image\n\n    Image.fromarray((np.asarray(mask, dtype=bool) * 255).astype(np.uint8), mode=\"L\").save(path)\n",
    "tools/build_benchmark.py": "\"\"\"Build the Phase 2 robustness benchmark: reference/target pairs on disk.\n\nThe benchmark asks one question with real data instead of synthetic edits: given\none annotated image of an object, how well does the pipeline find that same kind\nof object again in a photograph it has never seen? Every pair therefore carries\ntwo different photographs and two different masks, and the target mask is the\nonly thing used for scoring.\n\nCandidate sources were compared in `docs/dataset_comparison.md`. The conclusion\nis that COCO val2017 is the only public instance-mask corpus that is practical\nhere: its annotation file is small enough to fetch without downloading the whole\n241 MB archive, and its images can be pulled one file at a time. The cost is\nthat COCO does not track object identity across photographs, so a pair relates\ntwo *different instances of the same category*. Every sample therefore records\n`identity_level` so no result can be mistaken for same-object matching.\n`docs/pairing_protocol.md` explains the choice and the limits.\n\n    python tools/build_benchmark.py --pairs 100 --seed 0\n\nOutputs, all relative to `--root`:\n\n    raw/coco2017-val/*.jpg              cached source photographs\n    raw/annotations/instances_val2017.json\n    processed/source_index.json         which annotations were usable\n    samples/sample_0001/reference.jpg   the annotated object, first view\n    samples/sample_0001/reference_mask.png\n    samples/sample_0001/target.jpg      another photograph, same category\n    samples/sample_0001/target_mask.png\n    samples/sample_0001/metadata.json\n    splits/{train,validation,test}.json manifests for `evaluation.cases`\n\nThe split manifests use paths relative to their own directory, so\n`cases_from_manifest(\"data/splits/test.json\")` works from any working directory.\n\nSelecting annotations is deterministic: the same seed and the same annotation\nfile always yield the same pairs, which is what makes the saved results\nreproducible.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport argparse\nimport io\nimport json\nimport os\nimport random\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\nimport zipfile\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nCOCO_2017_VAL_ANNOTATIONS_URL = (\n    \"http://images.cocodataset.org/annotations/annotations_trainval2017.zip\"\n)\nCOCO_2017_VAL_ANNOTATION_MEMBER = \"annotations/instances_val2017.json\"\nCOCO_2017_VAL_IMAGE_URL = \"http://images.cocodataset.org/val2017/{file_name}\"\nDATASET_NAME = \"coco2017-val\"\n\nCONDITION_FIELDS = (\n    \"viewpoint_condition\",\n    \"illumination_condition\",\n    \"scale_condition\",\n    \"occlusion_condition\",\n)\nMETADATA_FIELDS = (\n    \"sample_id\",\n    \"object_id\",\n    \"dataset\",\n    \"category\",\n) + CONDITION_FIELDS\n\nSCALE_RATIO_SMALL = 0.5\nSCALE_RATIO_LARGE = 2.0\n\n\nclass HttpRangeFile(io.IOBase):\n    \"\"\"Read-only seekable view of a remote file, fetched one block at a time.\n\n    A ZIP archive is readable over HTTP without downloading all of it: the\n    central directory sits at the end and is small, so a seekable reader plus\n    `Range` requests are enough to pull out a single member. Requests here are\n    block-aligned and cached, which keeps the request count near the number of\n    blocks touched instead of the number of reads the ZIP parser performs.\n    \"\"\"\n\n    def __init__(self, url: str, block: int = 1 << 20, timeout: int = 60,\n                 retries: int = 3) -> None:\n        super().__init__()\n        self.url = url\n        self.block = int(block)\n        self.timeout = int(timeout)\n        self.retries = int(retries)\n        self.size = self._remote_size()\n        self._position = 0\n        self._buffer = b\"\"\n        self._buffer_start = 0\n\n    def _request(self, start: int, end: int) -> Tuple[bytes, Dict[str, str]]:\n        request = urllib.request.Request(self.url)\n        request.add_header(\"Range\", f\"bytes={start}-{end}\")\n        last: Optional[Exception] = None\n        for attempt in range(self.retries):\n            try:\n                with urllib.request.urlopen(request, timeout=self.timeout) as response:\n                    payload = response.read()\n                    headers = {key.lower(): value for key, value in response.headers.items()}\n                    return payload, headers\n            except (urllib.error.URLError, OSError, TimeoutError) as error:\n                last = error\n                time.sleep(1.0 + attempt)\n        raise IOError(f\"range request {start}-{end} failed for {self.url}: {last}\")\n\n    def _remote_size(self) -> int:\n        head = urllib.request.Request(self.url, method=\"HEAD\")\n        try:\n            with urllib.request.urlopen(head, timeout=self.timeout) as response:\n                length = response.headers.get(\"Content-Length\")\n                if length is not None:\n                    return int(length)\n        except (urllib.error.URLError, OSError, TimeoutError):\n            pass\n        payload, headers = self._request(0, 0)\n        del payload\n        content_range = headers.get(\"content-range\", \"\")\n        if \"/\" not in content_range:\n            raise IOError(f\"server did not report a size for {self.url}\")\n        return int(content_range.rsplit(\"/\", 1)[1])\n\n    def readable(self) -> bool:\n        return True\n\n    def seekable(self) -> bool:\n        return True\n\n    def writable(self) -> bool:\n        return False\n\n    def tell(self) -> int:\n        return self._position\n\n    def seek(self, offset: int, whence: int = io.SEEK_SET) -> int:\n        if whence == io.SEEK_SET:\n            position = offset\n        elif whence == io.SEEK_CUR:\n            position = self._position + offset\n        elif whence == io.SEEK_END:\n            position = self.size + offset\n        else:\n            raise ValueError(f\"unsupported whence {whence}\")\n        self._position = max(0, min(int(position), self.size))\n        return self._position\n\n    def read(self, size: int = -1) -> bytes:\n        if self._position >= self.size:\n            return b\"\"\n        remaining = self.size - self._position\n        wanted = remaining if size is None or size < 0 else min(int(size), remaining)\n        chunk = bytearray()\n        while len(chunk) < wanted:\n            start = self._position\n            in_buffer = (self._buffer_start <= start < self._buffer_start + len(self._buffer))\n            if in_buffer:\n                offset = start - self._buffer_start\n                take = min(wanted - len(chunk), len(self._buffer) - offset)\n                chunk.extend(self._buffer[offset:offset + take])\n                self._position = start + take\n                continue\n            block_start = start - (start % self.block)\n            block_end = min(block_start + self.block, self.size) - 1\n            payload, _ = self._request(block_start, block_end)\n            self._buffer = payload\n            self._buffer_start = block_start\n        return bytes(chunk)\n\n\ndef fetch_annotation_file(cache_path: str, url: str = COCO_2017_VAL_ANNOTATIONS_URL,\n                         member: str = COCO_2017_VAL_ANNOTATION_MEMBER,\n                         quiet: bool = False) -> str:\n    \"\"\"Return a local path to the COCO instance annotations, fetching if needed.\n\n    Only the requested member is transferred. The full archive is roughly 241 MB\n    while `instances_val2017.json` is about a fifth of that compressed, so this\n    saves the bulk of a slow download and leaves the rest of the archive alone.\n    \"\"\"\n    if os.path.exists(cache_path) and os.path.getsize(cache_path) > 1024:\n        return cache_path\n    os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)\n    if not quiet:\n        print(f\"fetching {member} from {url}\")\n    remote = HttpRangeFile(url)\n    if not quiet:\n        print(f\"archive {human(remote.size)}; reading central directory\")\n    with zipfile.ZipFile(remote) as archive:\n        info = archive.getinfo(member)\n        if not quiet:\n            print(f\"{member}: {human(info.compress_size)} compressed, \"\n                  f\"{human(info.file_size)} uncompressed\")\n        temporary = cache_path + \".part\"\n        started = time.time()\n        with archive.open(info) as source, open(temporary, \"wb\") as handle:\n            copied = 0\n            while True:\n                block = source.read(1 << 18)\n                if not block:\n                    break\n                handle.write(block)\n                copied += len(block)\n                if not quiet and copied % (4 << 20) < (1 << 18):\n                    rate = (copied / (1 << 20)) / max(time.time() - started, 1e-6)\n                    print(f\"  {human(copied)} at {rate:.2f} MB/s\")\n    os.replace(temporary, cache_path)\n    if not quiet:\n        print(f\"saved {human(os.path.getsize(cache_path))} to {cache_path}\")\n    return cache_path\n\n\ndef human(size: Optional[float]) -> str:\n    \"\"\"Format a byte count for progress output.\"\"\"\n    if size is None:\n        return \"unknown\"\n    value = float(size)\n    for unit in (\"B\", \"KB\", \"MB\", \"GB\"):\n        if value < 1024 or unit == \"GB\":\n            return f\"{value:.1f} {unit}\" if unit != \"B\" else f\"{int(value)} B\"\n        value /= 1024\n    return f\"{value:.1f} GB\"\n\n\ndef download_image(url: str, destination: str, quiet: bool = True) -> str:\n    \"\"\"Fetch one image, skipping the network entirely when it is already cached.\"\"\"\n    if os.path.exists(destination) and os.path.getsize(destination) > 512:\n        return destination\n    os.makedirs(os.path.dirname(destination), exist_ok=True)\n    request = urllib.request.Request(url, headers={\"User-Agent\": \"one-shot-benchmark/2.0\"})\n    last: Optional[Exception] = None\n    for attempt in range(3):\n        try:\n            with urllib.request.urlopen(request, timeout=60) as response:\n                payload = response.read()\n            if len(payload) < 512:\n                raise IOError(f\"short response ({len(payload)} bytes) for {url}\")\n            temporary = destination + \".part\"\n            with open(temporary, \"wb\") as handle:\n                handle.write(payload)\n            os.replace(temporary, destination)\n            if not quiet:\n                print(f\"    fetched {os.path.basename(destination)} ({human(len(payload))})\")\n            return destination\n        except (urllib.error.URLError, OSError, TimeoutError) as error:\n            last = error\n            time.sleep(1.0 + attempt)\n    raise IOError(f\"could not download {url}: {last}\")\n\n\ndef decode_compressed_counts(text: str) -> List[int]:\n    \"\"\"Decode COCO's string-encoded RLE counts into a list of run lengths.\n\n    COCO packs five six-bit symbols into one 32-bit integer, least significant\n    symbol first, with the sign carried in the lowest bit: a run whose length\n    occupies `n << 1` is a foreground run and `n << 1 | 1` is a background run.\n    Keeping this here means one dataset format detail does not pull in an extra\n    dependency for the notebook.\n    \"\"\"\n    if isinstance(text, list):\n        return [int(value) for value in text]\n    counts: List[int] = []\n    for index in range(0, len(text) - 1, 5):\n        chunk = text[index:index + 5]\n        if len(chunk) < 5:\n            break\n        value = 0\n        for shift in range(5):\n            digit = (ord(chunk[shift]) - 48) & 0x3F\n            value |= digit << (6 * shift)\n        counts.append(-((value - 1) >> 1) if value & 1 else value >> 1)\n    return counts\n\n\ndef encode_compressed_counts(counts: Sequence[int]) -> str:\n    \"\"\"Encode run lengths the way COCO stores them, the inverse of the decoder.\n\n    Only needed to prove the decoder against known values, and to write RLE\n    annotations back out if a later source uses them as the stored form.\n    \"\"\"\n    symbols: List[str] = []\n    for count in counts:\n        value = int(count)\n        encoded = ((-value) << 1 | 1) if value < 0 else (value << 1)\n        for shift in range(5):\n            symbols.append(chr(48 + ((encoded >> (6 * shift)) & 0x3F)))\n    return \"\".join(symbols)\n\n\ndef decode_rle(rle: Dict[str, Any]) -> np.ndarray:\n    \"\"\"Turn a COCO RLE into a boolean mask at the RLE's own resolution.\n\n    COCO scans columns top to bottom, so the flat mask is filled in `fortran`\n    order before being reshaped.\n    \"\"\"\n    height, width = int(rle[\"size\"][0]), int(rle[\"size\"][1])\n    counts = decode_compressed_counts(rle[\"counts\"])\n    flat = np.zeros(height * width, dtype=bool)\n    position = 0\n    value = False\n    for run in counts:\n        run = int(run)\n        if value and run > 0:\n            flat[position:position + run] = True\n        position += max(run, 0)\n        value = not value\n        if position >= flat.size:\n            break\n    return flat.reshape((height, width), order=\"F\")\n\n\ndef annotation_to_mask(segmentation: Dict[str, Any], height: int, width: int) -> np.ndarray:\n    \"\"\"Rasterise one COCO `segmentation` field at the image's resolution.\n\n    Polygons are filled with Pillow; RLE is decoded directly. Crowd regions are\n    never paired, so the union of a mask's parts is the object itself.\n    \"\"\"\n    from PIL import Image, ImageDraw\n\n    if isinstance(segmentation, list):\n        canvas = Image.new(\"L\", (width, height), 0)\n        draw = ImageDraw.Draw(canvas)\n        for polygon in segmentation:\n            points = [(float(polygon[i]), float(polygon[i + 1]))\n                      for i in range(0, len(polygon) - 1, 2)]\n            if len(points) >= 3:\n                draw.polygon(points, fill=255)\n        mask = np.asarray(canvas) > 127\n    else:\n        rle_mask = decode_rle(segmentation)\n        if rle_mask.shape != (height, width):\n            mask = np.asarray(Image.fromarray(rle_mask.astype(np.uint8) * 255)\n                              .resize((width, height), Image.NEAREST)) > 127\n        else:\n            mask = rle_mask\n    return mask\n\n\n@dataclass\nclass SourceObject:\n    \"\"\"One usable annotated instance from the source corpus.\"\"\"\n\n    annotation_id: int\n    image_id: int\n    file_name: str\n    category_id: int\n    category: str\n    area_fraction: float\n    bbox: Tuple[float, float, float, float]\n    width: int\n    height: int\n    segmentation: Dict[str, Any] = field(repr=False)\n\n\n@dataclass\nclass Pair:\n    \"\"\"A reference/target pair drawn from two different photographs.\"\"\"\n\n    reference: SourceObject\n    target: SourceObject\n    scale_condition: str\n    area_ratio: float\n\n\ndef load_source_objects(annotation_path: str, min_area_fraction: float, max_area_fraction: float,\n                        quiet: bool = False) -> Tuple[List[SourceObject], Dict[int, Dict[str, Any]]]:\n    \"\"\"Read the annotation file and keep the instances that can form a pair.\n\n    An instance qualifies when it is not a crowd region, occupies a workable\n    share of its photograph, and is not one of a handful of images that are\n    known to be unreadable in COCO val2017.\n    \"\"\"\n    with open(annotation_path, \"r\", encoding=\"utf-8\") as handle:\n        coco = json.load(handle)\n    images = {int(entry[\"id\"]): entry for entry in coco[\"images\"]}\n    categories = {int(entry[\"id\"]): entry for entry in coco[\"categories\"]}\n    objects: List[SourceObject] = []\n    for annotation in coco[\"annotations\"]:\n        if annotation.get(\"iscrowd\", 0):\n            continue\n        image = images.get(int(annotation[\"image_id\"]))\n        if image is None or not image.get(\"file_name\"):\n            continue\n        height, width = int(image[\"height\"]), int(image[\"width\"])\n        if height <= 0 or width <= 0:\n            continue\n        fraction = float(annotation[\"area\"]) / float(height * width)\n        if not min_area_fraction <= fraction <= max_area_fraction:\n            continue\n        segmentation = annotation.get(\"segmentation\")\n        if not segmentation:\n            continue\n        objects.append(\n            SourceObject(\n                annotation_id=int(annotation[\"id\"]),\n                image_id=int(annotation[\"image_id\"]),\n                file_name=str(image[\"file_name\"]),\n                category_id=int(annotation[\"category_id\"]),\n                category=str(categories[int(annotation[\"category_id\"])][\"name\"]),\n                area_fraction=fraction,\n                bbox=tuple(float(value) for value in annotation[\"bbox\"]),  # type: ignore[arg-type]\n                width=width,\n                height=height,\n                segmentation=segmentation,\n            )\n        )\n    objects.sort(key=lambda item: (item.category_id, item.image_id, item.annotation_id))\n    if not quiet:\n        print(f\"usable instances: {len(objects)} across \"\n              f\"{len({item.category_id for item in objects})} categories\")\n    return objects, images\n\n\ndef classify_scale(reference: SourceObject, target: SourceObject) -> Tuple[str, float]:\n    \"\"\"Name the scale relation from measured mask areas, not from annotations.\"\"\"\n    ratio = target.area_fraction / max(reference.area_fraction, 1e-9)\n    if ratio < SCALE_RATIO_SMALL:\n        return \"smaller_in_target\", ratio\n    if ratio > SCALE_RATIO_LARGE:\n        return \"larger_in_target\", ratio\n    return \"similar\", ratio\n\n\ndef build_pairs(objects: Sequence[SourceObject], count: int, seed: int,\n                categories_cap: Optional[int] = None,\n                quiet: bool = False) -> List[Pair]:\n    \"\"\"Choose pairs deterministically, spreading them over as many categories as possible.\n\n    Category balance comes first: COCO is dominated by `person`, so a naive sample\n    would be almost entirely people. Each category contributes a near-equal share\n    before any category is allowed a second pair, and both photographs of a pair\n    are always different images.\n    \"\"\"\n    by_category: Dict[int, List[SourceObject]] = {}\n    for item in objects:\n        by_category.setdefault(item.category_id, []).append(item)\n\n    usable: List[Tuple[int, List[SourceObject]]] = []\n    for category_id, items in sorted(by_category.items()):\n        image_ids = {item.image_id for item in items}\n        if len(image_ids) >= 2:\n            usable.append((category_id, items))\n    usable.sort(key=lambda entry: (-len({item.image_id for item in entry[1]}), entry[0]))\n    if categories_cap is not None:\n        usable = usable[:categories_cap]\n    if not usable:\n        raise SystemExit(\"no category has instances in two different images\")\n\n    rng = random.Random(seed)\n    queues: List[List[SourceObject]] = []\n    for _, items in usable:\n        shuffled = list(items)\n        rng.shuffle(shuffled)\n        queues.append(shuffled)\n\n    pairs: List[Pair] = []\n    exhausted: List[int] = []\n    while len(pairs) < count and len(exhausted) < len(queues):\n        for index in range(len(queues)):\n            if index in exhausted:\n                continue\n            queue = queues[index]\n            if len(queue) < 2:\n                exhausted.append(index)\n                continue\n            reference, target = queue.pop(), queue.pop()\n            while reference.image_id == target.image_id and queue:\n                target = queue.pop()\n            if reference.image_id == target.image_id:\n                exhausted.append(index)\n                continue\n            condition, ratio = classify_scale(reference, target)\n            pairs.append(Pair(reference=reference, target=target,\n                              scale_condition=condition, area_ratio=ratio))\n            if len(pairs) >= count:\n                break\n    if not quiet:\n        spread: Dict[str, int] = {}\n        for pair in pairs:\n            spread[pair.reference.category] = spread.get(pair.reference.category, 0) + 1\n        print(f\"selected {len(pairs)} pairs over {len(spread)} categories \"\n              f\"(requested {count}, seed {seed})\")\n    return pairs\n\n\ndef write_sample(pair: Pair, index: int, root: str, images_dir: str,\n                 quiet: bool = False) -> Dict[str, Any]:\n    \"\"\"Download both photographs, render both masks, and write one sample folder.\n\n    Image bytes are cached under `raw/` and hard-linked into the sample folder,\n    so a benchmark never stores the same photograph twice on disk while each\n    sample still stands on its own.\n    \"\"\"\n    from PIL import Image\n\n    sample_id = f\"sample_{index:04d}\"\n    directory = os.path.join(root, \"samples\", sample_id)\n    os.makedirs(directory, exist_ok=True)\n\n    paths: Dict[str, str] = {}\n    masks: Dict[str, np.ndarray] = {}\n    for role, item in ((\"reference\", pair.reference), (\"target\", pair.target)):\n        cached = os.path.join(images_dir, item.file_name)\n        download_image(COCO_2017_VAL_IMAGE_URL.format(file_name=item.file_name), cached, quiet)\n        image_path = os.path.join(directory, f\"{role}.jpg\")\n        try:\n            os.link(cached, image_path)\n        except (OSError, AttributeError):\n            with open(cached, \"rb\") as source, open(image_path, \"wb\") as handle:\n                handle.write(source.read())\n        paths[f\"{role}_image\"] = os.path.relpath(image_path, root).replace(os.sep, \"/\")\n        mask = annotation_to_mask(item.segmentation, item.height, item.width)\n        if not mask.any():\n            raise ValueError(f\"empty rendered mask for {role} of {sample_id}\")\n        masks[role] = mask\n        mask_path = os.path.join(directory, f\"{role}_mask.png\")\n        Image.fromarray((mask * 255).astype(np.uint8), mode=\"L\").save(mask_path)\n        paths[f\"{role}_mask\"] = os.path.relpath(mask_path, root).replace(os.sep, \"/\")\n\n    with Image.open(os.path.join(directory, \"reference.jpg\")) as probe:\n        reference_size = probe.size\n    with Image.open(os.path.join(directory, \"target.jpg\")) as probe:\n        target_size = probe.size\n\n    metadata: Dict[str, Any] = {\n        \"sample_id\": sample_id,\n        \"object_id\": f\"coco2017-{pair.reference.annotation_id}-{pair.target.annotation_id}\",\n        \"identity_level\": \"category\",\n        \"dataset\": DATASET_NAME,\n        \"category\": pair.reference.category,\n        \"category_id\": pair.reference.category_id,\n        \"reference_image\": paths[\"reference_image\"],\n        \"reference_mask\": paths[\"reference_mask\"],\n        \"target_image\": paths[\"target_image\"],\n        \"target_mask\": paths[\"target_mask\"],\n        \"viewpoint_condition\": \"unknown\",\n        \"illumination_condition\": \"unknown\",\n        \"scale_condition\": pair.scale_condition,\n        \"occlusion_condition\": \"unknown\",\n        \"measured\": {\n            \"reference_area_fraction\": round(float(masks[\"reference\"].mean()), 6),\n            \"target_area_fraction\": round(float(masks[\"target\"].mean()), 6),\n            \"area_ratio_target_over_reference\": round(float(pair.area_ratio), 4),\n            \"reference_image_size\": list(reference_size),\n            \"target_image_size\": list(target_size),\n            \"reference_bbox\": [round(value, 2) for value in pair.reference.bbox],\n            \"target_bbox\": [round(value, 2) for value in pair.target.bbox],\n        },\n        \"source\": {\n            \"image_id\": [pair.reference.image_id, pair.target.image_id],\n            \"annotation_id\": [pair.reference.annotation_id, pair.target.annotation_id],\n            \"file_name\": [pair.reference.file_name, pair.target.file_name],\n            \"annotation_file\": \"instances_val2017.json\",\n        },\n    }\n    with open(os.path.join(directory, \"metadata.json\"), \"w\", encoding=\"utf-8\") as handle:\n        json.dump(metadata, handle, indent=2, sort_keys=True)\n        handle.write(\"\\n\")\n    return metadata\n\n\ndef split_samples(entries: Sequence[Dict[str, Any]], seed: int,\n                  ratios: Tuple[float, float, float] = (0.6, 0.2, 0.2)) -> Dict[str, List[Dict[str, Any]]]:\n    \"\"\"Partition samples into train/validation/test with a seeded shuffle.\"\"\"\n    ordered = sorted(entries, key=lambda item: item[\"sample_id\"])\n    rng = random.Random(seed + 1)\n    rng.shuffle(ordered)\n    total = len(ordered)\n    train_end = int(round(total * ratios[0]))\n    validation_end = train_end + int(round(total * ratios[1]))\n    buckets = {\n        \"train\": ordered[:train_end],\n        \"validation\": ordered[train_end:validation_end],\n        \"test\": ordered[validation_end:],\n    }\n    return buckets\n\n\ndef write_splits(buckets: Dict[str, List[Dict[str, Any]]], root: str, seed: int,\n                 quiet: bool = False) -> Dict[str, str]:\n    \"\"\"Write one manifest per split, in the schema `evaluation.cases` already reads.\"\"\"\n    directory = os.path.join(root, \"splits\")\n    os.makedirs(directory, exist_ok=True)\n    written: Dict[str, str] = {}\n    for name, entries in buckets.items():\n        cases = []\n        for metadata in entries:\n            measured = metadata.get(\"measured\", {})\n            cases.append({\n                \"name\": metadata[\"sample_id\"],\n                \"reference_image\": f\"../{metadata['reference_image']}\",\n                \"reference_mask\": f\"../{metadata['reference_mask']}\",\n                \"target_image\": f\"../{metadata['target_image']}\",\n                \"target_mask\": f\"../{metadata['target_mask']}\",\n                \"object_id\": metadata[\"object_id\"],\n                \"dataset\": metadata[\"dataset\"],\n                \"category\": metadata[\"category\"],\n                \"identity_level\": metadata[\"identity_level\"],\n                \"reference_area_fraction\": measured.get(\"reference_area_fraction\"),\n                \"target_area_fraction\": measured.get(\"target_area_fraction\"),\n                **{field: metadata[field] for field in CONDITION_FIELDS},\n            })\n        manifest = {\n            \"dataset\": DATASET_NAME,\n            \"split\": name,\n            \"seed\": seed,\n            \"pairing\": \"class-level: two different instances of the same category in \"\n                       \"two different photographs\",\n            \"cases\": cases,\n        }\n        path = os.path.join(directory, f\"{name}.json\")\n        with open(path, \"w\", encoding=\"utf-8\") as handle:\n            json.dump(manifest, handle, indent=2)\n            handle.write(\"\\n\")\n        written[name] = path\n        if not quiet:\n            print(f\"  {name}: {len(cases)} cases -> {path}\")\n    return written\n\n\ndef build(root: str, pairs: int, seed: int, min_area_fraction: float, max_area_fraction: float,\n          categories_cap: Optional[int], annotation_path: Optional[str], annotation_url: str,\n          dry_run: bool) -> Dict[str, Any]:\n    \"\"\"Run the whole build: annotations, pair selection, samples, splits.\"\"\"\n    quiet = False\n    annotation_cache = annotation_path or os.path.join(\n        root, \"raw\", \"annotations\", \"instances_val2017.json\")\n    if not os.path.exists(annotation_cache):\n        fetch_annotation_file(annotation_cache, url=annotation_url, quiet=quiet)\n    objects, _ = load_source_objects(annotation_cache, min_area_fraction, max_area_fraction, quiet)\n    selection = build_pairs(objects, pairs, seed, categories_cap, quiet)\n\n    report: Dict[str, Any] = {\n        \"dataset\": DATASET_NAME,\n        \"seed\": seed,\n        \"requested_pairs\": pairs,\n        \"annotation_file\": os.path.relpath(annotation_cache, root).replace(os.sep, \"/\"),\n        \"min_area_fraction\": min_area_fraction,\n        \"max_area_fraction\": max_area_fraction,\n        \"pairs\": [],\n    }\n    if dry_run:\n        for pair in selection:\n            report[\"pairs\"].append({\n                \"reference_image\": pair.reference.file_name,\n                \"target_image\": pair.target.file_name,\n                \"category\": pair.reference.category,\n                \"scale_condition\": pair.scale_condition,\n            })\n        print(json.dumps(report, indent=2))\n        return report\n\n    images_dir = os.path.join(root, \"raw\", \"coco2017-val\")\n    entries: List[Dict[str, Any]] = []\n    started = time.time()\n    for index, pair in enumerate(selection, start=1):\n        metadata = write_sample(pair, index, root, images_dir, quiet=False)\n        entries.append(metadata)\n        report[\"pairs\"].append({\n            \"sample_id\": metadata[\"sample_id\"],\n            \"category\": metadata[\"category\"],\n            \"scale_condition\": metadata[\"scale_condition\"],\n        })\n        print(f\"[{index:3d}/{len(selection)}] {metadata['sample_id']} \"\n              f\"{metadata['category']:<14s} {metadata['scale_condition']:<18s} \"\n              f\"area x{metadata['measured']['area_ratio_target_over_reference']:.2f}\")\n\n    buckets = split_samples(entries, seed)\n    write_splits(buckets, root, seed, quiet)\n\n    index_path = os.path.join(root, \"processed\", \"source_index.json\")\n    os.makedirs(os.path.dirname(index_path), exist_ok=True)\n    report[\"samples\"] = len(entries)\n    report[\"categories\"] = sorted({metadata[\"category\"] for metadata in entries})\n    report[\"elapsed_seconds\"] = round(time.time() - started, 1)\n    with open(index_path, \"w\", encoding=\"utf-8\") as handle:\n        json.dump(report, handle, indent=2)\n        handle.write(\"\\n\")\n    print(f\"built {len(entries)} samples in {report['elapsed_seconds']}s \"\n          f\"over {len(report['categories'])} categories\")\n    print(f\"source index -> {index_path}\")\n    return report\n\n\ndef main(argv: Optional[Sequence[str]] = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0],\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument(\"--root\", default=\"data\", help=\"dataset root (default: data)\")\n    parser.add_argument(\"--pairs\", type=int, default=100, help=\"number of pairs to build\")\n    parser.add_argument(\"--seed\", type=int, default=0, help=\"deterministic selection seed\")\n    parser.add_argument(\"--min-area-fraction\", type=float, default=0.005,\n                        help=\"drop instances smaller than this share of the image\")\n    parser.add_argument(\"--max-area-fraction\", type=float, default=0.6,\n                        help=\"drop instances larger than this share of the image\")\n    parser.add_argument(\"--categories-cap\", type=int, default=None,\n                        help=\"use at most this many categories\")\n    parser.add_argument(\"--annotations\", default=None,\n                        help=\"use an existing instances_val2017.json instead of fetching\")\n    parser.add_argument(\"--annotations-url\", default=COCO_2017_VAL_ANNOTATIONS_URL)\n    parser.add_argument(\"--dry-run\", action=\"store_true\", help=\"select pairs and print them only\")\n    arguments = parser.parse_args(argv)\n    if arguments.pairs < 1:\n        parser.error(\"--pairs must be positive\")\n    build(root=arguments.root, pairs=arguments.pairs, seed=arguments.seed,\n          min_area_fraction=arguments.min_area_fraction,\n          max_area_fraction=arguments.max_area_fraction,\n          categories_cap=arguments.categories_cap,\n          annotation_path=arguments.annotations,\n          annotation_url=arguments.annotations_url,\n          dry_run=arguments.dry_run)\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
    "tools/run_benchmark.py": "\"\"\"Run the pipeline over a benchmark split and save results, figures and provenance.\n\nThis is the Phase 2 counterpart to `evaluation.cases.run_cases`, which printed a\nline per case. A benchmark has to survive being interrupted, has to keep its\nnumbers next to the machine that produced them, and has to write figures that\nexplain the numbers. So each sample is scored into its own JSON file first, and\nthe tables and figures are assembled from those files afterwards.\n\nThat ordering is what makes resume cheap and safe: a completed sample is never\nrecomputed, and if the process dies halfway the next run picks up from the last\nfinished sample. Deleting a sample's JSON is how you ask for it to be redone.\n\n    python tools/run_benchmark.py --split test --device cuda --figures 12\n    python tools/run_benchmark.py --split test --resume          # fill in what is missing\n\nOutputs under `--output` (default `results/baseline`):\n\n    samples/<sample_id>.json    one record per pair, the unit of resume\n    results.csv                 one row per pair, the table of record\n    summary.json                aggregate statistics per metric and group\n    iou_distribution.png        run-level figures\n    iou_by_category.png\n    runtime_by_stage.png\n    provenance.json             seed, model ids, versions, GPU, dataset, split\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport argparse\nimport os\nimport sys\nimport time\nfrom dataclasses import dataclass\nfrom typing import Any, Dict, List, Optional, Sequence\n\nimport numpy as np\n\nsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nfrom evaluation.cases import cases_from_manifest  # noqa: E402\nfrom evaluation.dataset_integrity import check_dataset  # noqa: E402\nfrom evaluation.metrics import aggregate_rows, evaluate_mask, iou_histogram  # noqa: E402\nfrom evaluation.runtime import (  # noqa: E402\n    build_provenance,\n    describe_environment,\n    reset_gpu_memory,\n    set_seed,\n    summarize_times,\n    write_json,\n)\n\nCSV_COLUMNS = (\n    \"sample_id\", \"object_id\", \"method\", \"iou\", \"dice\", \"precision\", \"recall\",\n    \"runtime\", \"feature_time\", \"segmentation_time\", \"found\", \"confidence\",\n    \"dataset\", \"category\", \"identity_level\",\n    \"viewpoint_condition\", \"illumination_condition\", \"scale_condition\", \"occlusion_condition\",\n    \"reference_area_fraction\", \"target_area_fraction\", \"peak_gpu_mb\", \"device\",\n)\n\n\n@dataclass\nclass BenchmarkOptions:\n    \"\"\"Everything a run needs to know, in one place, for the record.\"\"\"\n\n    root: str = \"data\"\n    split: str = \"test\"\n    output: str = \"results/baseline\"\n    method: str = \"dinov2-base+sam-base\"\n    device: Optional[str] = None\n    encoder_model: str = \"facebook/dinov2-base\"\n    segmenter_model: str = \"facebook/sam-vit-base\"\n    seed: int = 0\n    limit: Optional[int] = None\n    figures: int = 8\n    resume: bool = True\n    force: bool = False\n    strict_integrity: bool = False\n\n\ndef _stage_seconds(result: Any) -> Dict[str, float]:\n    runtime = getattr(result, \"runtime\", {}) or {}\n    stages: Dict[str, float] = {}\n    for name, value in runtime.items():\n        if isinstance(value, (int, float)) and not isinstance(value, bool):\n            stages[name] = float(value)\n    return stages\n\n\ndef _csv_row(record: Dict[str, Any]) -> Dict[str, Any]:\n    row = {column: record.get(column) for column in CSV_COLUMNS}\n    stages = record.get(\"stage_seconds\") or {}\n    row[\"runtime\"] = stages.get(\"total\")\n    row[\"feature_time\"] = stages.get(\"features\")\n    row[\"segmentation_time\"] = stages.get(\"segmentation\")\n    return row\n\n\ndef write_csv(path: str, rows: Sequence[Dict[str, Any]]) -> str:\n    \"\"\"Write the results table with a fixed column order.\"\"\"\n    import csv\n\n    directory = os.path.dirname(os.path.abspath(path))\n    if directory:\n        os.makedirs(directory, exist_ok=True)\n    with open(path, \"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(CSV_COLUMNS), extrasaction=\"ignore\")\n        writer.writeheader()\n        for row in rows:\n            formatted = {}\n            for column in CSV_COLUMNS:\n                value = row.get(column)\n                if isinstance(value, float):\n                    formatted[column] = f\"{value:.6g}\"\n                elif isinstance(value, (bool, np.bool_)):\n                    formatted[column] = int(value)\n                elif value is None:\n                    formatted[column] = \"\"\n                else:\n                    formatted[column] = value\n            writer.writerow(formatted)\n    return path\n\n\ndef run_split(options: BenchmarkOptions, pipeline: Any = None,\n              cases: Optional[Sequence[Any]] = None) -> Dict[str, Any]:\n    \"\"\"Score one split and write every artefact. Returns the summary dict.\n\n    The pipeline is built once and reused, which is what makes the per-sample\n    timings comparable: they all pay the same one-off model load.\n    \"\"\"\n    from pipeline.one_shot_segmentation import get_default_pipeline\n\n    set_seed(options.seed)\n    manifest = os.path.join(options.root, \"splits\", f\"{options.split}.json\")\n    if options.split == \"all\":\n        manifests = [os.path.join(options.root, \"splits\", f\"{name}.json\")\n                     for name in (\"train\", \"validation\", \"test\")]\n        missing = [path for path in manifests if not os.path.isfile(path)]\n        if missing:\n            raise SystemExit(f\"split manifest(s) not found: {missing}\\n\"\n                             f\"build the benchmark first: python tools/build_benchmark.py\")\n    elif not os.path.isfile(manifest):\n        raise SystemExit(f\"split manifest not found: {manifest}\\n\"\n                         f\"build the benchmark first: python tools/build_benchmark.py\")\n    report = check_dataset(root=options.root, strict=options.strict_integrity)\n    if not report.ok():\n        raise SystemExit(\"dataset integrity check failed, refusing to run:\\n\"\n                         + report.summary(limit=8))\n    print(f\"integrity: {report.samples_checked} samples, \"\n          f\"{len(report.categories)} categories, no critical findings\")\n\n    if cases is None:\n        if options.split == \"all\":\n            cases = []\n            seen: set = set()\n            for path in manifests:\n                for case in cases_from_manifest(path):\n                    if case.name not in seen:\n                        seen.add(case.name)\n                        cases.append(case)\n            manifest = \", \".join(os.path.basename(path) for path in manifests)\n        else:\n            cases = cases_from_manifest(manifest)\n    if options.limit is not None:\n        cases = list(cases)[:options.limit]\n    print(f\"split {options.split}: {len(cases)} cases from {manifest}\")\n\n    if pipeline is None:\n        print(f\"loading models (device={options.device or 'auto'})\")\n        pipeline = get_default_pipeline(\n            device=options.device,\n            encoder_kwargs={\"model_id\": options.encoder_model},\n            segmenter_kwargs={\"model_id\": options.segmenter_model},\n        )\n\n    samples_dir = os.path.join(options.output, \"samples\")\n    os.makedirs(samples_dir, exist_ok=True)\n    records: List[Dict[str, Any]] = []\n    reused = 0\n    started = time.time()\n\n    for position, case in enumerate(cases, start=1):\n        record_path = os.path.join(samples_dir, f\"{case.name}.json\")\n        if options.resume and not options.force and os.path.isfile(record_path):\n            try:\n                import json\n\n                with open(record_path, \"r\", encoding=\"utf-8\") as handle:\n                    cached = json.load(handle)\n                records.append(cached)\n                reused += 1\n                continue\n            except (OSError, ValueError):\n                pass\n\n        reset_gpu_memory(getattr(pipeline, \"device\", None))\n        sample_started = time.perf_counter()\n        result = pipeline.segment_object(case.reference_image, case.reference_mask,\n                                         case.target_image)\n        wall = time.perf_counter() - sample_started\n\n        if case.scorable:\n            metrics = evaluate_mask(result.predicted_mask, case.target_mask)\n            metric_values: Dict[str, Any] = metrics.to_dict()\n            metric_values[\"iou\"] = float(metrics.iou)\n        else:\n            metric_values = {name: None for name in (\"iou\", \"dice\", \"precision\", \"recall\")}\n\n        stages = _stage_seconds(result)\n        record: Dict[str, Any] = {\n            \"sample_id\": case.name,\n            \"object_id\": case.meta.get(\"object_id\"),\n            \"method\": options.method,\n            \"dataset\": case.meta.get(\"dataset\"),\n            \"category\": case.meta.get(\"category\"),\n            \"identity_level\": case.meta.get(\"identity_level\"),\n            \"viewpoint_condition\": case.meta.get(\"viewpoint_condition\", \"unknown\"),\n            \"illumination_condition\": case.meta.get(\"illumination_condition\", \"unknown\"),\n            \"scale_condition\": case.meta.get(\"scale_condition\", \"unknown\"),\n            \"occlusion_condition\": case.meta.get(\"occlusion_condition\", \"unknown\"),\n            \"reference_area_fraction\": case.meta.get(\"reference_area_fraction\"),\n            \"target_area_fraction\": case.meta.get(\"target_area_fraction\"),\n            \"found\": bool(result.found),\n            \"confidence\": float(result.confidence),\n            \"wall_seconds\": float(wall),\n            \"stage_seconds\": stages,\n            \"peak_gpu_mb\": _peak_memory(pipeline),\n            \"device\": str(getattr(pipeline, \"device\", \"unknown\")),\n            **metric_values,\n        }\n        write_json(record_path, record)\n        records.append(record)\n\n        iou = record.get(\"iou\")\n        iou_text = \"  n/a\" if iou is None else f\"IoU {iou:.3f}\"\n        print(f\"[{position:3d}/{len(cases)}] {case.name:14s} {str(case.meta.get('category')):<14s} \"\n              f\"{iou_text:12s} conf {record['confidence']:.3f} {wall:6.2f}s\")\n\n    records.sort(key=lambda item: str(item.get(\"sample_id\")))\n    scored = [record for record in records if record.get(\"iou\") is not None]\n    grouped = aggregate_rows(scored, by=(\"category\", \"scale_condition\", \"identity_level\"))\n    grouped[\"iou_histogram\"] = iou_histogram([record[\"iou\"] for record in scored])\n    grouped[\"runtime\"] = {\n        \"wall\": summarize_times([record[\"wall_seconds\"] for record in records]),\n        \"per_stage\": {\n            stage: summarize_times([record[\"stage_seconds\"][stage] for record in records\n                                    if stage in (record.get(\"stage_seconds\") or {})])\n            for stage in (\"features\", \"correspondence\", \"localization\", \"segmentation\")\n        },\n    }\n    grouped[\"found_rate\"] = (\n        float(np.mean([record[\"found\"] for record in records])) if records else 0.0\n    )\n    grouped[\"split\"] = options.split\n    grouped[\"method\"] = options.method\n    grouped[\"samples\"] = len(records)\n    grouped[\"scored\"] = len(scored)\n    grouped[\"reused_from_cache\"] = reused\n    grouped[\"elapsed_seconds\"] = round(time.time() - started, 1)\n\n    csv_path = write_csv(os.path.join(options.output, \"results.csv\"), [_csv_row(record) for record in records])\n    write_json(os.path.join(options.output, \"summary.json\"), grouped)\n    write_json(os.path.join(options.output, \"provenance.json\"), build_provenance(\n        pipeline=pipeline, device=getattr(pipeline, \"device\", None), seed=options.seed,\n        dataset={\n            \"root\": options.root,\n            \"split\": options.split,\n            \"manifest\": manifest,\n            \"samples\": len(records),\n            \"dataset\": records[0].get(\"dataset\") if records else None,\n            \"identity_level\": records[0].get(\"identity_level\") if records else None,\n        },\n        extra={\"method\": options.method, \"elapsed_seconds\": grouped[\"elapsed_seconds\"]},\n    ))\n\n    figures = _write_figures(records, options, grouped)\n    grouped[\"artefacts\"] = {\"csv\": csv_path, \"figures\": figures}\n    print(f\"\\nwrote {csv_path}\")\n    print(f\"scored {len(scored)}/{len(records)} pairs, found rate {grouped['found_rate'] * 100:.0f}%, \"\n          f\"reused {reused} cached, {grouped['elapsed_seconds']}s\")\n    if scored:\n        overall = grouped[\"overall\"][\"iou\"]\n        print(f\"IoU mean {overall['mean']:.3f} median {overall['median']:.3f} \"\n              f\"std {overall['std']:.3f} | >=0.5 {overall['success_rate_at_0_5'] * 100:.0f}% \"\n              f\">=0.7 {overall['success_rate_at_0_7'] * 100:.0f}%\")\n    return grouped\n\n\ndef _peak_memory(pipeline: Any) -> Optional[float]:\n    from evaluation.runtime import gpu_memory_mb\n\n    return gpu_memory_mb(getattr(pipeline, \"device\", None))\n\n\ndef _write_figures(records: Sequence[Dict[str, Any]], options: BenchmarkOptions,\n                   grouped: Dict[str, Any]) -> Dict[str, str]:\n    \"\"\"Save the run-level figures. Never allowed to lose the numbers.\"\"\"\n    from visualization.benchmark import plot_benchmark_summary\n\n    directory = os.path.join(options.output, \"figures\")\n    os.makedirs(directory, exist_ok=True)\n    written: Dict[str, str] = {}\n    try:\n        written.update(plot_benchmark_summary(records, directory, method=options.method))\n    except Exception as error:  # noqa: BLE001 - figures must not lose the numbers\n        print(f\"warning: run-level figures failed: {error}\")\n    return written\n\n\ndef save_pair_figures(options: BenchmarkOptions, pipeline: Any = None,\n                      limit: Optional[int] = None, worst: bool = True) -> List[str]:\n    \"\"\"Re-run pairs to draw the seven-panel figure, worst first.\n\n    Separate from the scoring loop on purpose: figures are the expensive optional\n    part, and a reader usually wants the failures rather than an arbitrary few.\n    \"\"\"\n    import matplotlib\n\n    matplotlib.use(\"Agg\")\n\n    from pipeline.one_shot_segmentation import get_default_pipeline\n    from visualization.benchmark import save_benchmark_pair\n\n    names = (\"train\", \"validation\", \"test\") if options.split == \"all\" else (options.split,)\n    cases: List[Any] = []\n    seen: set = set()\n    for name in names:\n        path = os.path.join(options.root, \"splits\", f\"{name}.json\")\n        if not os.path.isfile(path):\n            continue\n        for case in cases_from_manifest(path):\n            if case.name not in seen:\n                seen.add(case.name)\n                cases.append(case)\n    if not cases:\n        raise SystemExit(f\"no cases found under {options.root}/splits for {options.split}\")\n    records = []\n    for record_dir in sorted(os.listdir(os.path.join(options.output, \"samples\"))):\n        if not record_dir.endswith(\".json\"):\n            continue\n        import json\n\n        with open(os.path.join(options.output, \"samples\", record_dir), \"r\", encoding=\"utf-8\") as handle:\n            records.append(json.load(handle))\n    scored = [record for record in records if record.get(\"iou\") is not None]\n    scored.sort(key=lambda item: float(item[\"iou\"]))\n    chosen = scored if worst else scored[::-1]\n    if limit is not None:\n        chosen = chosen[:limit]\n\n    by_name = {case.name: case for case in cases}\n    if pipeline is None:\n        pipeline = get_default_pipeline(\n            device=options.device,\n            encoder_kwargs={\"model_id\": options.encoder_model},\n            segmenter_kwargs={\"model_id\": options.segmenter_model},\n        )\n    directory = os.path.join(options.output, \"figures\")\n    written: List[str] = []\n    for record in chosen:\n        case = by_name.get(str(record[\"sample_id\"]))\n        if case is None:\n            continue\n        result = pipeline.segment_object(case.reference_image, case.reference_mask,\n                                         case.target_image)\n        metrics = evaluate_mask(result.predicted_mask, case.target_mask) if case.scorable else None\n        caption = (f\"{case.name} | {case.meta.get('category')} | \"\n                   f\"scale: {case.meta.get('scale_condition')} | \"\n                   f\"identity: {case.meta.get('identity_level')}\")\n        written.append(save_benchmark_pair(\n            os.path.join(directory, f\"{case.name}.png\"),\n            reference_image=case.reference_image,\n            reference_mask=case.reference_mask,\n            target_image=case.target_image,\n            similarity_map=result.similarity_map,\n            candidate_region=result.candidate_region,\n            predicted_mask=result.predicted_mask,\n            target_mask=case.target_mask,\n            metrics=metrics,\n            caption=caption,\n            confidence=float(result.confidence),\n        ))\n        print(f\"figure {written[-1]}\")\n    return written\n\n\ndef main(argv: Optional[Sequence[str]] = None) -> int:\n    import matplotlib\n\n    matplotlib.use(\"Agg\")\n\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0],\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument(\"--root\", default=\"data\", help=\"dataset root (default: data)\")\n    parser.add_argument(\"--split\", default=\"test\",\n                        choices=(\"train\", \"validation\", \"test\", \"all\"),\n                        help=\"which split to score; 'all' concatenates the three\")\n    parser.add_argument(\"--output\", default=None, help=\"results directory (default results/<split>)\")\n    parser.add_argument(\"--method\", default=\"dinov2-base+sam-base\")\n    parser.add_argument(\"--device\", default=None, help=\"cuda, cpu, or omit for auto\")\n    parser.add_argument(\"--encoder-model\", default=\"facebook/dinov2-base\")\n    parser.add_argument(\"--segmenter-model\", default=\"facebook/sam-vit-base\")\n    parser.add_argument(\"--seed\", type=int, default=0)\n    parser.add_argument(\"--limit\", type=int, default=None, help=\"score only the first N cases\")\n    parser.add_argument(\"--figures\", type=int, default=8,\n                        help=\"draw per-pair figures for the N worst pairs (0 to skip)\")\n    parser.add_argument(\"--no-resume\", action=\"store_true\", help=\"recompute every sample\")\n    parser.add_argument(\"--strict-integrity\", action=\"store_true\",\n                        help=\"treat integrity warnings as failures\")\n    arguments = parser.parse_args(argv)\n\n    output = arguments.output or os.path.join(\"results\", arguments.split)\n    options = BenchmarkOptions(\n        root=arguments.root, split=arguments.split, output=output, method=arguments.method,\n        device=arguments.device, encoder_model=arguments.encoder_model,\n        segmenter_model=arguments.segmenter_model, seed=arguments.seed, limit=arguments.limit,\n        figures=arguments.figures, resume=not arguments.no_resume,\n        strict_integrity=arguments.strict_integrity,\n    )\n    environment = describe_environment(arguments.device)\n    print(f\"device: {environment.get('device')} | cuda: {environment.get('cuda_available')} \"\n          f\"| git: {str(environment.get('git_revision'))[:8]}\")\n    run_split(options)\n    if options.figures:\n        save_pair_figures(options, limit=options.figures, worst=True)\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
    "visualization/benchmark.py": "\"\"\"Benchmark figures: one row per pair, plus the distribution across a run.\n\n`visualization/results.py` shows what a single run did. A benchmark needs two\nmore views. `plot_benchmark_pair` puts the whole story of one pair side by side\nin the order a reader needs it: what was shown as the reference, what was being\nlooked for, what the encoder thought matched, which region was chosen, what came\nout, and how that compares with the truth. `plot_benchmark_summary` covers a whole\nrun: the IoU distribution, mean IoU per category, and runtime per stage, which is\nwhere a regression shows up first.\n\nTwo deliberate choices are worth stating. Every panel is captioned with the value\nthat produced it, so a figure stays readable without the CSV beside it. And\nfailures are drawn, not hidden: a run that found nothing still gets a full row\nwith an explicit \"no candidate\" caption, because those rows are the interesting\nones.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport os\nfrom typing import Any, Dict, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom evaluation.metrics import MaskMetrics\nfrom visualization.results import _overlay_mask, _show_image, _to_rgb\n\n\ndef _figure(nrows: int, ncols: int, width: float = 3.1, height: float = 2.9):\n    import matplotlib.pyplot as plt\n\n    figure, axes = plt.subplots(nrows, ncols, figsize=(width * ncols, height * nrows),\n                               squeeze=False)\n    return figure, axes\n\n\ndef _title(axis, text: str, colour: str = \"black\", size: float = 9.0) -> None:\n    axis.set_title(text, fontsize=size, color=colour, pad=3)\n\n\ndef _mask_panel(axis, mask: Optional[np.ndarray], colour: str = \"#ff2d55\") -> None:\n    \"\"\"Show a mask on its own, as a flat silhouette.\"\"\"\n    if mask is None or not np.asarray(mask).any():\n        axis.text(0.5, 0.5, \"no prediction\", ha=\"center\", va=\"center\",\n                  transform=axis.transAxes, fontsize=9, color=\"#b0b0b0\")\n        axis.set_axis_off()\n        return\n    binary = np.asarray(mask, dtype=bool)\n    rgba = np.zeros((*binary.shape, 4), dtype=np.float32)\n    rgb = np.array(_to_rgb(colour), dtype=np.float32) / 255.0\n    rgba[..., :3] = rgb\n    rgba[..., 3] = binary.astype(np.float32)\n    axis.imshow(rgba)\n    axis.set_axis_off()\n\n\ndef _similarity_panel(axis, similarity: Optional[np.ndarray]) -> None:\n    if similarity is None or not np.asarray(similarity).size:\n        axis.text(0.5, 0.5, \"no map\", ha=\"center\", va=\"center\", transform=axis.transAxes,\n                  fontsize=9, color=\"#b0b0b0\")\n        axis.set_axis_off()\n        return\n    axis.imshow(np.asarray(similarity, dtype=float), cmap=\"magma\", vmin=0.0, vmax=1.0)\n    axis.set_axis_off()\n\n\ndef plot_benchmark_pair(\n    reference_image: Any,\n    reference_mask: np.ndarray,\n    target_image: Any,\n    similarity_map: Optional[np.ndarray] = None,\n    candidate_region: Optional[np.ndarray] = None,\n    predicted_mask: Optional[np.ndarray] = None,\n    target_mask: Optional[np.ndarray] = None,\n    metrics: Optional[MaskMetrics] = None,\n    caption: str = \"\",\n    confidence: Optional[float] = None,\n    figsize: Optional[Tuple[float, float]] = None,\n):\n    \"\"\"Draw the seven panels that explain one pair, left to right.\n\n    Args:\n        reference_image: the image the object was shown from.\n        reference_mask: the annotated reference mask.\n        target_image: the image the object was to be found in.\n        similarity_map: (H, W) correspondence scores in [0, 1].\n        candidate_region: the region localisation selected.\n        predicted_mask: the final mask, or `None` when nothing was found.\n        target_mask: the ground truth for the target, when it exists.\n        metrics: the scored metrics, shown on the overlay panel.\n        caption: dataset, category and condition labels for the title.\n        confidence: the pipeline's confidence, shown on the candidate panel.\n    \"\"\"\n    figure, axes = _figure(1, 7, width=3.0, height=3.1)\n    if figsize is not None:\n        figure.set_size_inches(*figsize)\n\n    _show_image(axes[0][0], reference_image)\n    _title(axes[0][0], f\"reference image\\n{reference_mask.mean() * 100:.1f}% of frame\")\n\n    _mask_panel(axes[0][1], reference_mask, colour=\"#2f7dff\")\n    _title(axes[0][1], \"reference mask\\nthe query object\")\n\n    _show_image(axes[0][2], target_image)\n    _title(axes[0][2], \"target image\\nunseen photograph\")\n\n    _similarity_panel(axes[0][3], similarity_map)\n    if similarity_map is not None and np.asarray(similarity_map).size:\n        _title(axes[0][3], f\"similarity map\\npeak {float(np.max(similarity_map)):.2f}\")\n    else:\n        _title(axes[0][3], \"similarity map\")\n\n    _mask_panel(axes[0][4], candidate_region, colour=\"#f5a623\")\n    _title(axes[0][4],\n           f\"candidate region\\nconf {confidence:.3f}\" if confidence is not None\n           else \"candidate region\\nnone found\")\n\n    _mask_panel(axes[0][5], predicted_mask, colour=\"#00c853\")\n    found = predicted_mask is not None and bool(np.asarray(predicted_mask).any())\n    _title(axes[0][5], f\"predicted mask\\n{found * np.asarray(predicted_mask).mean() * 100:.1f}% of frame\"\n           if found else \"predicted mask\\nNOT FOUND\",\n           colour=\"black\" if found else \"#d32f2f\", size=8.5)\n\n    _overlay_mask(axes[0][6], target_image, target_mask, colour=\"#00c853\", alpha=0.55)\n    if predicted_mask is not None and np.asarray(predicted_mask).any():\n        _overlay_mask(axes[0][6], target_image, predicted_mask, colour=\"#ff2d55\", alpha=0.45)\n    if metrics is not None:\n        _title(axes[0][6],\n               f\"truth green / pred red\\nIoU {metrics.iou:.3f}  Dice {metrics.dice:.3f}\\n\"\n               f\"P {metrics.precision:.3f}  R {metrics.recall:.3f}\",\n               colour=\"black\" if metrics.iou >= 0.5 else \"#d32f2f\", size=8.5)\n    else:\n        _title(axes[0][6], \"no ground truth\")\n\n    if caption:\n        figure.suptitle(caption, fontsize=10, y=0.99)\n    figure.tight_layout(rect=(0, 0, 1, 0.94 if caption else 1.0))\n    return figure\n\n\ndef save_benchmark_pair(path: str, **kwargs: Any) -> str:\n    \"\"\"Render one pair figure and write it to `path`.\"\"\"\n    figure = plot_benchmark_pair(**kwargs)\n    directory = os.path.dirname(os.path.abspath(path))\n    if directory:\n        os.makedirs(directory, exist_ok=True)\n    figure.savefig(path, dpi=140, bbox_inches=\"tight\")\n    import matplotlib.pyplot as plt\n\n    plt.close(figure)\n    return path\n\n\ndef plot_iou_distribution(values: Sequence[float], bins: int = 10, title: str = \"IoU distribution\",\n                          path: Optional[str] = None):\n    \"\"\"Histogram of IoU with success-rate markers, as one axis.\"\"\"\n    import matplotlib.pyplot as plt\n\n    figure, axis = plt.subplots(figsize=(7.2, 3.6))\n    array = np.asarray([float(value) for value in values if value is not None], dtype=float)\n    if array.size == 0:\n        axis.text(0.5, 0.5, \"no scored samples\", ha=\"center\", va=\"center\", transform=axis.transAxes)\n        axis.set_axis_off()\n    else:\n        axis.hist(np.clip(array, 0.0, 1.0), bins=bins, range=(0.0, 1.0),\n                  color=\"#2f7dff\", edgecolor=\"white\", linewidth=0.8)\n        for threshold, colour in ((0.5, \"#2e7d32\"), (0.7, \"#f9a825\"), (0.9, \"#c62828\")):\n            share = float((array >= threshold).mean())\n            axis.axvline(threshold, color=colour, linestyle=\"--\", linewidth=1.0, alpha=0.8)\n            axis.text(threshold, axis.get_ylim()[1] * 0.97, f\"  IoU>={threshold:g}: {share * 100:.0f}%\",\n                      color=colour, fontsize=8, va=\"top\")\n        axis.set_xlabel(\"IoU between predicted and ground-truth mask\")\n        axis.set_ylabel(\"pairs\")\n        axis.set_xlim(0.0, 1.0)\n        axis.set_title(f\"{title}  (n={array.size}, mean {array.mean():.3f}, \"\n                      f\"median {np.median(array):.3f})\", fontsize=10)\n        axis.grid(axis=\"y\", alpha=0.25, linewidth=0.6)\n    figure.tight_layout()\n    if path is not None:\n        directory = os.path.dirname(os.path.abspath(path))\n        if directory:\n            os.makedirs(directory, exist_ok=True)\n        figure.savefig(path, dpi=140, bbox_inches=\"tight\")\n        plt.close(figure)\n        return path\n    return figure\n\n\ndef plot_group_means(rows: Sequence[Dict[str, Any]], column: str = \"category\",\n                     metric: str = \"iou\", max_groups: int = 30, path: Optional[str] = None):\n    \"\"\"Mean metric per group, sorted worst first, with counts in the tick labels.\"\"\"\n    import matplotlib.pyplot as plt\n\n    buckets: Dict[str, list] = {}\n    for row in rows:\n        if metric in row and row[metric] is not None:\n            buckets.setdefault(str(row.get(column, \"unknown\")), []).append(float(row[metric]))\n    if not buckets:\n        return plot_iou_distribution([], path=path)\n    ordered = sorted(buckets.items(), key=lambda item: float(np.mean(item[1])))[:max_groups]\n    labels = [f\"{name}\\n(n={len(values)})\" for name, values in ordered]\n    means = [float(np.mean(values)) for _, values in ordered]\n    spread = [float(np.std(values)) for _, values in ordered]\n\n    figure, axis = plt.subplots(figsize=(max(7.0, 0.42 * len(labels)), 4.0))\n    positions = np.arange(len(labels))\n    axis.bar(positions, means, yerr=spread, color=\"#00acc1\", edgecolor=\"white\", linewidth=0.7,\n             capsize=2)\n    axis.axhline(float(np.mean([row[metric] for row in rows if metric in row\n                                and row[metric] is not None])), color=\"#455a64\",\n                 linestyle=\"--\", linewidth=1.0, label=\"overall mean\")\n    axis.set_xticks(positions)\n    axis.set_xticklabels(labels, rotation=60, ha=\"right\", fontsize=7)\n    axis.set_ylabel(f\"mean {metric}\")\n    axis.set_ylim(0.0, 1.0)\n    axis.set_title(f\"mean {metric} by {column} (worst {len(ordered)} of {len(buckets)})\", fontsize=10)\n    axis.legend(fontsize=8, loc=\"upper right\")\n    axis.grid(axis=\"y\", alpha=0.25, linewidth=0.6)\n    figure.tight_layout()\n    if path is not None:\n        directory = os.path.dirname(os.path.abspath(path))\n        if directory:\n            os.makedirs(directory, exist_ok=True)\n        figure.savefig(path, dpi=140, bbox_inches=\"tight\")\n        plt.close(figure)\n        return path\n    return figure\n\n\ndef plot_stage_runtime(rows: Sequence[Dict[str, Any]], path: Optional[str] = None):\n    \"\"\"Where the time went: median seconds per pipeline stage, with a per-sample spread.\"\"\"\n    import matplotlib.pyplot as plt\n\n    stages: Dict[str, list] = {}\n    for row in rows:\n        for name, value in (row.get(\"stage_seconds\") or {}).items():\n            if isinstance(value, (int, float)):\n                stages.setdefault(name, []).append(float(value))\n    if not stages:\n        return None\n    ordered = sorted(stages.items(), key=lambda item: float(np.median(item[1])), reverse=True)\n    labels = [name.replace(\"_\", \" \") for name, _ in ordered]\n    medians = [float(np.median(values)) for _, values in ordered]\n\n    figure, axis = plt.subplots(figsize=(7.0, 0.6 * len(labels) + 1.6))\n    positions = np.arange(len(labels))\n    axis.barh(positions, medians, color=\"#7e57c2\", edgecolor=\"white\", linewidth=0.7)\n    axis.set_yticks(positions)\n    axis.set_yticklabels(labels, fontsize=8)\n    axis.invert_yaxis()\n    axis.set_xlabel(\"median seconds per pair\")\n    axis.set_title(\"pipeline cost by stage\", fontsize=10)\n    for index, (_, values) in enumerate(ordered):\n        axis.text(medians[index], index, f\"  p95 {np.percentile(values, 95):.2f}s\",\n                  va=\"center\", fontsize=7, color=\"#424242\")\n    axis.grid(axis=\"x\", alpha=0.25, linewidth=0.6)\n    figure.tight_layout()\n    if path is not None:\n        directory = os.path.dirname(os.path.abspath(path))\n        if directory:\n            os.makedirs(directory, exist_ok=True)\n        figure.savefig(path, dpi=140, bbox_inches=\"tight\")\n        plt.close(figure)\n        return path\n    return figure\n\n\ndef plot_benchmark_summary(rows: Sequence[Dict[str, Any]], directory: str,\n                           method: str = \"baseline\") -> Dict[str, str]:\n    \"\"\"Write the three run-level figures and return their paths.\"\"\"\n    values = [row[\"iou\"] for row in rows if row.get(\"iou\") is not None]\n    written = {\n        \"iou_distribution\": plot_iou_distribution(\n            values, title=f\"IoU distribution, {method}\", path=os.path.join(directory, \"iou_distribution.png\")),\n        \"per_category\": plot_group_means(\n            rows, column=\"category\", metric=\"iou\",\n            path=os.path.join(directory, \"iou_by_category.png\")) or \"\",\n        \"runtime\": plot_stage_runtime(\n            rows, path=os.path.join(directory, \"runtime_by_stage.png\")) or \"\",\n    }\n    return written\n",
    "visualization/results.py": "\"\"\"Step 8 - visualising a one-shot result.\n\n`plot_one_shot_result` renders a single row of panels that tell the whole story\nof a run: the reference image with its mask, the similarity map, the localised\ncandidates with their boxes, the final mask, and an optional ground-truth\noverlay for the evaluation examples. `save_result_figure` writes the same figure\nto disk, and `plot_feature_grid` shows where the encoder's patches actually sit,\nwhich is the quickest way to see whether the grid geometry is sane.\n\"\"\"\n\nfrom __future__ import annotations\n\nNOTEBOOK_SOURCE_REVISION = '6e041d5bbedfb2d5'  # injected by tools/embed_sources.py\n\nimport os\nfrom typing import Any, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom models.feature_encoder import ImageInput, to_pil_image\n\n\ndef _figure(nrows: int = 1, ncols: int = 1, figsize: Optional[Tuple[float, float]] = None):\n    import matplotlib.pyplot as plt\n\n    return plt.subplots(nrows, ncols, figsize=figsize or (4.0 * ncols, 3.4 * nrows), squeeze=False)\n\n\ndef _show_image(axis, image: ImageInput) -> None:\n    axis.imshow(np.asarray(to_pil_image(image)))\n    axis.set_axis_off()\n\n\ndef _overlay_mask(\n    axis,\n    image: ImageInput,\n    mask: Optional[np.ndarray],\n    colour: str = \"#ff2d55\",\n    alpha: float = 0.45,\n) -> None:\n    array = np.asarray(to_pil_image(image))\n    axis.imshow(array)\n    if mask is not None and np.asarray(mask).any():\n        coloured = np.zeros((*np.asarray(mask).shape, 4), dtype=np.float32)\n        rgb = np.array(_to_rgb(colour), dtype=np.float32) / 255.0\n        coloured[..., :3] = rgb\n        coloured[..., 3] = np.asarray(mask, dtype=np.float32) * alpha\n        axis.imshow(coloured)\n    axis.set_axis_off()\n\n\ndef _to_rgb(colour: str) -> Tuple[int, int, int]:\n    from matplotlib.colors import to_rgb\n\n    return tuple(int(channel * 255) for channel in to_rgb(colour))\n\n\ndef plot_one_shot_result(\n    reference_image: ImageInput,\n    reference_mask: np.ndarray,\n    target_image: ImageInput,\n    result: Any,\n    ground_truth: Optional[np.ndarray] = None,\n    figsize: Optional[Tuple[float, float]] = None,\n    show: bool = True,\n):\n    \"\"\"Draw the panels for one `OneShotResult` and return the matplotlib figure.\"\"\"\n    ncols = 5 if ground_truth is not None else 4\n    figure, axes = _figure(1, ncols, figsize=figsize or (4.0 * ncols, 3.6))\n\n    _overlay_mask(axes[0][0], reference_image, reference_mask, colour=\"#00d1b2\")\n    axes[0][0].set_title(\"reference + mask\", fontsize=10)\n\n    _show_image(axes[0][1], target_image)\n    axes[0][1].set_title(\"target\", fontsize=10)\n\n    similarity = np.asarray(result.similarity_map, dtype=np.float32)\n    handle = axes[0][2].imshow(similarity, cmap=\"magma\", vmin=0.0, vmax=1.0)\n    axes[0][2].set_axis_off()\n    axes[0][2].set_title(f\"similarity (peak {similarity.max():.2f})\", fontsize=10)\n    figure.colorbar(handle, ax=axes[0][2], fraction=0.046, pad=0.04)\n\n    _show_image(axes[0][3], target_image)\n    candidates = list(getattr(result, \"candidates\", ()) or ())\n    for rank, candidate in enumerate(candidates):\n        x1, y1, x2, y2 = candidate.box\n        axes[0][3].add_patch(\n            _rectangle((x1, y1), x2 - x1, y2 - y1, edge=\"#ffd166\" if rank else \"#ff2d55\", label=f\"C{rank + 1}\")\n        )\n    if getattr(result, \"candidate\", None) is not None:\n        x1, y1, x2, y2 = result.candidate.box\n        axes[0][3].add_patch(_rectangle((x1, y1), x2 - x1, y2 - y1, edge=\"#ff2d55\", label=\"selected\"))\n    confidence = float(getattr(result, \"confidence\", 0.0))\n    axes[0][3].set_title(f\"candidates (conf {confidence:.2f})\", fontsize=10)\n    if candidates:\n        axes[0][3].legend(loc=\"lower right\", fontsize=7)\n\n    if ground_truth is not None:\n        _overlay_mask(axes[0][4], target_image, result.predicted_mask, colour=\"#ff2d55\", alpha=0.45)\n        truth_box = _bounding_box(ground_truth)\n        if truth_box is not None:\n            x1, y1, x2, y2 = truth_box\n            axes[0][4].add_patch(_rectangle((x1, y1), x2 - x1, y2 - y1, edge=\"#22c55e\", label=\"ground truth\"))\n        if result.predicted_mask is not None:\n            predicted = np.asarray(result.predicted_mask, dtype=bool)\n            truth = np.asarray(ground_truth, dtype=bool)\n            intersection = float(np.logical_and(predicted, truth).sum())\n            union = float(np.logical_or(predicted, truth).sum())\n            iou = intersection / union if union else 0.0\n        else:\n            iou = 0.0\n        axes[0][4].set_title(f\"prediction (magenta) vs GT (green) - IoU {iou:.2f}\", fontsize=10)\n        axes[0][4].legend(loc=\"lower right\", fontsize=7)\n\n    if getattr(result, \"predicted_mask\", None) is None:\n        for axis in axes[0][3:]:\n            axis.text(\n                0.5, 0.5, \"no candidate\", ha=\"center\", va=\"center\", color=\"white\",\n                transform=axis.transAxes, fontsize=9,\n            )\n\n    runtime = getattr(result, \"runtime\", {}) or {}\n    figure.suptitle(\n        f\"one-shot segmentation - total {float(runtime.get('total', 0.0)):.2f} s\",\n        fontsize=11,\n    )\n    figure.tight_layout(rect=(0, 0, 1, 0.95))\n    if show:\n        import matplotlib.pyplot as plt\n\n        plt.show()\n    return figure\n\n\ndef _rectangle(origin: Tuple[float, float], width: float, height: float, edge: str, label: str = \"\"):\n    from matplotlib.patches import Rectangle\n\n    return Rectangle(origin, width, height, fill=False, edgecolor=edge, linewidth=2.0, label=label)\n\n\ndef _bounding_box(mask: np.ndarray) -> Optional[Tuple[int, int, int, int]]:\n    ys, xs = np.nonzero(np.asarray(mask, dtype=bool))\n    if xs.size == 0:\n        return None\n    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1\n\n\ndef plot_feature_grid(\n    features: Any,\n    image: Optional[ImageInput] = None,\n    figsize: Tuple[float, float] = (9.0, 4.2),\n    show: bool = True,\n):\n    \"\"\"Overlay the encoder patch grid on the image to check the geometry.\"\"\"\n    figure, axes = _figure(1, 2, figsize=figsize)\n    if image is not None:\n        _show_image(axes[0][0], image)\n        axes[0][0].set_title(\"input\", fontsize=10)\n    grid_h, grid_w = features.grid_size\n    height, width = features.input_size\n    for row in range(grid_h + 1):\n        y = row * height / grid_h\n        axes[0][1].plot([0, width], [y, y], color=\"#38bdf8\", linewidth=0.4)\n    for col in range(grid_w + 1):\n        x = col * width / grid_w\n        axes[0][1].plot([x, x], [0, height], color=\"#38bdf8\", linewidth=0.4)\n    axes[0][1].set_xlim(0, width)\n    axes[0][1].set_ylim(height, 0)\n    axes[0][1].set_axis_off()\n    axes[0][1].set_title(f\"patch grid {grid_w} x {grid_h} (patch {features.patch_size}px)\", fontsize=10)\n    figure.tight_layout()\n    if show:\n        import matplotlib.pyplot as plt\n\n        plt.show()\n    return figure\n\n\ndef save_result_figure(\n    path: str,\n    reference_image: ImageInput,\n    reference_mask: np.ndarray,\n    target_image: ImageInput,\n    result: Any,\n    ground_truth: Optional[np.ndarray] = None,\n    dpi: int = 140,\n) -> str:\n    \"\"\"Render the result panels to `path` (png, pdf, ...).\"\"\"\n    directory = os.path.dirname(os.path.abspath(path))\n    if directory:\n        os.makedirs(directory, exist_ok=True)\n    figure = plot_one_shot_result(\n        reference_image,\n        reference_mask,\n        target_image,\n        result,\n        ground_truth=ground_truth,\n        show=False,\n    )\n    figure.savefig(path, dpi=dpi, bbox_inches=\"tight\")\n    import matplotlib.pyplot as plt\n\n    plt.close(figure)\n    return os.path.abspath(path)\n\n\ndef plot_similarity_map(similarity_map: np.ndarray, title: str = \"similarity map\", show: bool = True):\n    \"\"\"Standalone heatmap of the similarity map.\"\"\"\n    figure, axes = _figure(1, 1, figsize=(4.4, 3.8))\n    handle = axes[0][0].imshow(np.asarray(similarity_map, dtype=np.float32), cmap=\"magma\", vmin=0.0, vmax=1.0)\n    axes[0][0].set_axis_off()\n    axes[0][0].set_title(title, fontsize=10)\n    figure.colorbar(handle, ax=axes[0][0], fraction=0.046, pad=0.04)\n    figure.tight_layout()\n    if show:\n        import matplotlib.pyplot as plt\n\n        plt.show()\n    return figure\n"
}

for relative, source in SOURCE_FILES.items():
    path = os.path.join(BUNDLE_ROOT, relative)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        handle.write(source)

sys.path.insert(0, os.path.abspath(BUNDLE_ROOT))
print(f"wrote {len(SOURCE_FILES)} modules into {os.path.abspath(BUNDLE_ROOT)} (revision {EXPECTED_SOURCE_REVISION})")

## Check the loaded code

The cell above rewrites the source files, but Python keeps the modules it already
imported, so a session can go on executing the previous code while the files on disk
are new - and the resulting traceback quotes line numbers from two versions of the
same file. This cell compares the revision of every imported module against the one
that was just written and reloads the mismatched ones in dependency order.

A successful reload prints `reloaded N stale module(s)`, and the session continues
normally. If a reload is impossible the cell stops with a message that starts with
`Fix this first: Runtime > Restart session, then Run all`.

In [ ]:
import importlib

# Dependency order: a module is reloaded only after everything it imports.
CHECKED = (
    "models.feature_encoder",
    "models.segmenter",
    "matching.reference_representation",
    "matching.correspondence",
    "matching.localization",
    "pipeline.one_shot_segmentation",
    "evaluation.metrics",
    "evaluation.cases",
    "evaluation.dataset_integrity",
    "evaluation.runtime",
    "visualization.results",
    "visualization.benchmark",

)


def loaded_revisions():
    revisions = {}
    for name in CHECKED:
        module = importlib.import_module(name)
        revisions[name] = getattr(module, "NOTEBOOK_SOURCE_REVISION", None)
    return revisions


def describe(name, revision):
    if revision is None:
        return f"  {name}: imported before this notebook revision existed"
    return f"  {name}: {revision} (expected {EXPECTED_SOURCE_REVISION})"


def reload_stale(revisions):
    """Re-execute the mismatched modules so this session picks up the new code.

    Reloading works here because nothing has been built yet: the source cell only
    writes files, and this cell runs before any model, image or result exists, so
    there is no object still holding a reference to the old classes.
    """
    for name in CHECKED:
        if revisions.get(name) == EXPECTED_SOURCE_REVISION:
            continue
        try:
            importlib.reload(importlib.import_module(name))
        except Exception as error:
            raise RuntimeError(
                f"Fix this first: Runtime > Restart session, then Run all.\n"
                f"Reloading {name} failed ({type(error).__name__}: {error})."
            ) from error


revisions = loaded_revisions()
stale = {name: revision for name, revision in revisions.items() if revision != EXPECTED_SOURCE_REVISION}
if stale:
    print("stale modules in this session:")
    for name, revision in stale.items():
        print(describe(name, revision))
    reload_stale(revisions)
    revisions = loaded_revisions()
    still_stale = {name: revision for name, revision in revisions.items()
                   if revision != EXPECTED_SOURCE_REVISION}
    if still_stale:
        raise RuntimeError(
            "Fix this first: Runtime > Restart session, then Run all.\n"
            "This session still runs different code than the cell above wrote:\n"
            + "\n".join(describe(name, revision) for name, revision in still_stale.items())
        )
    print(f"reloaded {len(stale)} stale module(s); the session now matches the source")

print(f"loaded modules match the embedded source (revision {EXPECTED_SOURCE_REVISION})")

## 2. Where the data lives

The benchmark data is large and deliberately not committed, so it is looked up
rather than assumed. On Colab, run the next cell to mount Drive and point
`OSB_PROJECT_DIR` at the copy of this repository; locally the same variables
are read from the environment.

    OSB_PROJECT_DIR   repository root (data/, results/ live underneath)
    OSB_DATA_ROOT     override the data root
    OSB_RESULTS_DIR   override where results are written
    OSB_SPLIT         train | validation | test | all   (default all)
    OSB_LIMIT         run only the first N pairs        (default all)
    OSB_SEED          benchmark seed                    (default 0)
    OSB_PAIR_FIGURES  best and worst pairs drawn inline (default 3)

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive")
except ImportError:
    print("not running in Colab - Drive is not mounted, local paths are used")

In [ ]:
import os
import sys

def find_project():
    candidates = [
        os.environ.get("OSB_PROJECT_DIR", ""),
        os.getcwd(),
        "/content/drive/MyDrive/one-shot-segmentation",
        "/content/drive/MyDrive/Nouveau dossier",
        "/content/one-shot-segmentation",
    ]
    for candidate in candidates:
        if candidate and os.path.isdir(os.path.join(candidate, "data", "samples")):
            return candidate
    for candidate in candidates:
        if candidate and os.path.isdir(os.path.join(candidate, "data")):
            return candidate
    return os.getcwd()

PROJECT_DIR = find_project()
DATA_ROOT = os.environ.get("OSB_DATA_ROOT", os.path.join(PROJECT_DIR, "data"))
RESULTS_DIR = os.environ.get("OSB_RESULTS_DIR", os.path.join(PROJECT_DIR, "results", "baseline"))
SPLIT = os.environ.get("OSB_SPLIT", "all")
SEED = int(os.environ.get("OSB_SEED", "0"))
PAIRS = int(os.environ.get("OSB_PAIRS", "100"))
LIMIT = int(os.environ.get("OSB_LIMIT", "0") or 0) or None
PAIR_FIGURES = int(os.environ.get("OSB_PAIR_FIGURES", "3"))
IN_COLAB = os.path.isdir("/content/drive")

print(f"project : {PROJECT_DIR}")
print(f"data    : {DATA_ROOT}  exists={os.path.isdir(DATA_ROOT)}")
print(f"results : {RESULTS_DIR}")
print(f"split={SPLIT} seed={SEED} limit={LIMIT} pairs={PAIRS}")

from evaluation import dataset_integrity as _integrity

print("modules from:", os.path.dirname(os.path.dirname(_integrity.__file__)))

## 3. Integrity gate

Nothing runs against unverified data. The check reports missing or corrupt
images and masks, empty or mismatched masks, duplicate pairs, split leakage and
split coverage, with every finding tagged `critical`, `warning` or `info`. It
runs before the models are loaded so a broken dataset costs seconds, not an
hour of inference.

`scripts/check_dataset.py --root data` runs the same gate from a terminal.

In [ ]:
from evaluation.dataset_integrity import check_dataset

report = check_dataset(root=DATA_ROOT, strict=False)
print(report.summary(limit=20))
if not report.ok():
    raise RuntimeError("critical integrity findings - fix the data before benchmarking")

## 4. Pair set

The pair set is built once and then reused, so a rerun never silently selects
different objects. Building needs the COCO 2017 annotation file (fetched over
HTTP Range from the 241 MB zip if it is not cached) and the images referenced
by the chosen pairs; both land under `data/raw/` and are hard-linked into the
samples, so images are stored once.

In [ ]:
import json
import os

samples_dir = os.path.join(DATA_ROOT, "samples")
existing = sorted(name for name in os.listdir(samples_dir)) if os.path.isdir(samples_dir) else []

if len(existing) < PAIRS:
    from tools.build_benchmark import build, COCO_2017_VAL_ANNOTATIONS_URL

    print(f"building {PAIRS} pairs into {DATA_ROOT} ...")
    built = build(root=DATA_ROOT, pairs=PAIRS, seed=SEED, min_area_fraction=0.005,
                  max_area_fraction=0.6, categories_cap=None, annotation_path=None,
                  annotation_url=COCO_2017_VAL_ANNOTATIONS_URL, dry_run=False)
    print(f"built {built.get('written', len(existing))} samples")
else:
    print(f"{len(existing)} samples already present in {samples_dir}")

for split in ("train", "validation", "test"):
    path = os.path.join(DATA_ROOT, "splits", f"{split}.json")
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        cases = payload.get("cases", [])
        print(f"{split:12s} {len(cases):4d} pairs")
    else:
        print(f"{split:12s} MISSING - build the benchmark first")

## 5. Load models

The same frozen `facebook/dinov2-base` encoder and `facebook/sam-vit-base`
segmenter as Phase 1: `eval()`, `requires_grad_(False)`, no optimiser, no
fine-tuning. The pipeline object is built here once and handed to the runner so
the pair figures below reuse it instead of loading the checkpoints twice.

In [ ]:
import torch

from pipeline.one_shot_segmentation import get_default_pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

pipeline = get_default_pipeline(
    device=DEVICE,
    encoder_kwargs={"model_id": "facebook/dinov2-base"},
    segmenter_kwargs={"model_id": "facebook/sam-vit-base"},
)
print("pipeline ready:", type(pipeline).__name__)

## 6. Run the benchmark

`run_split` re-runs the integrity gate, resolves the manifest(s), skips any
sample whose JSON record already exists (resume, on by default), scores the
rest, and writes `results.csv`, `summary.json`, `provenance.json` and the
run-level figures. Set `OSB_LIMIT=2` before running the notebook to smoke-test
the whole path in a couple of minutes.

Each record carries the condition columns - `dataset`, `identity_level`,
`scale_condition`, `category`, `split` - next to the metrics, because a number
without its condition is not interpretable for this question.

In [ ]:
from tools.run_benchmark import BenchmarkOptions, run_split

options = BenchmarkOptions(
    root=DATA_ROOT,
    split=SPLIT,
    output=RESULTS_DIR,
    device=DEVICE,
    seed=SEED,
    limit=LIMIT,
    resume=True,
    figures=0,
)

summary = run_split(options, pipeline=pipeline)
print(f"scored {summary.get('scored')} / {summary.get('samples')} pairs, "
      f"{summary.get('reused_from_cache')} reused from cache")

## 7. Results

The headline numbers, then the conditions they depend on. `identity_level` is
the honesty column: COCO pairs are `category` transfer - the reference and the
target show two *different* objects of the same class - so a low IoU on a
category like `person` means the frozen pipeline does not transfer, not that a
viewpoint change was simulated.

In [ ]:
import json as _json
import os

with open(os.path.join(RESULTS_DIR, "summary.json"), "r", encoding="utf-8") as handle:
    stored = _json.load(handle)

overall = stored.get("overall", {}).get("iou", {})
print("overall IoU")
for key in ("count", "mean", "median", "std", "min", "max",
            "success_rate_at_0_5", "success_rate_at_0_7"):
    if key in overall:
        print(f"  {key:22s} {overall[key]:.4f}" if isinstance(overall[key], float)
              else f"  {key:22s} {overall[key]}")
print(f"  {'found rate':22s} {stored.get('found_rate', 0.0):.4f}")

for column, label in (("scale_condition", "scale condition"),
                      ("identity_level", "identity level"),
                      ("category", "category")):
    groups = stored.get("groups", {}).get(column) or {}
    if not groups:
        continue
    print(f"\n{label} (IoU mean / median / n)")
    for key, value in sorted(groups.items(), key=lambda item: item[1].get("iou", {}).get("mean", -1),
                             reverse=True):
        stats = value.get("iou", {})
        print(f"  {key:24s} {stats.get('mean', float('nan')):6.3f} "
              f"{stats.get('median', float('nan')):6.3f}  n={value.get('count')}")

## 8. Figures

Run-level figures are written to disk by the runner; the cell below re-draws
them inline from the per-sample records. Two more cells draw individual pairs
panel by panel, which is where the result is legible: reference mask, target
prediction, target ground truth, similarity map, and the stages in between.
One cell takes the lowest IoU pairs, the other the highest, and both save their
panels into `results/figures/` so a run on Colab leaves the images behind.

In [ ]:
import json as _json
import os

import matplotlib.pyplot as plt
from visualization.benchmark import (
    plot_group_means,
    plot_iou_distribution,
    plot_stage_runtime,
)

rows = []
samples_dir = os.path.join(RESULTS_DIR, "samples")
for name in sorted(os.listdir(samples_dir)) if os.path.isdir(samples_dir) else []:
    if name.endswith(".json"):
        with open(os.path.join(samples_dir, name), "r", encoding="utf-8") as handle:
            rows.append(_json.load(handle))

values = [row["iou"] for row in rows if row.get("iou") is not None]
print(f"{len(values)} scored pairs of {len(rows)} records")

fig = plot_iou_distribution(values, title=f"IoU distribution, {stored.get('method', 'baseline')}")
plt.show()

fig = plot_group_means(rows, column="scale_condition", metric="iou")
plt.show()

fig = plot_group_means(rows, column="category", metric="iou")
plt.show()

fig = plot_stage_runtime(rows)
plt.show()

The worst pairs. This re-runs the pipeline on purpose: the per-sample records
hold scores and timings, not the images needed to draw a panel, and the count
stays small so a Colab CPU session can finish it.

In [ ]:
import json as _json
import os

import matplotlib.pyplot as plt

from evaluation.cases import cases_from_manifest
from evaluation.metrics import evaluate_mask
from visualization.benchmark import plot_benchmark_pair

manifests = [os.path.join(DATA_ROOT, "splits", f"{name}.json")
             for name in ("train", "validation", "test")]
cases, seen = [], set()
for path in manifests:
    if os.path.isfile(path):
        for case in cases_from_manifest(path):
            if case.name not in seen:
                seen.add(case.name)
                cases.append(case)

records = []
for case in cases:
    path = os.path.join(RESULTS_DIR, "samples", f"{case.name}.json")
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as handle:
            records.append(_json.load(handle))

scored = sorted((record for record in records if record.get("iou") is not None),
                key=lambda record: float(record["iou"]))
worst = scored[:PAIR_FIGURES]
print(f"drawing the {len(worst)} worst of {len(scored)} scored pairs")

by_name = {case.name: case for case in cases}
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
for record in worst:
    case = by_name[str(record["sample_id"])]
    result = pipeline.segment_object(case.reference_image, case.reference_mask,
                                     case.target_image)
    metrics = evaluate_mask(result.predicted_mask, case.target_mask) if case.scorable else None
    caption = (f"{case.name} | {case.meta.get('category')} | "
               f"scale: {case.meta.get('scale_condition')} | "
               f"identity: {case.meta.get('identity_level')}")
    figure = plot_benchmark_pair(
        reference_image=case.reference_image,
        reference_mask=case.reference_mask,
        target_image=case.target_image,
        similarity_map=result.similarity_map,
        candidate_region=result.candidate_region,
        predicted_mask=result.predicted_mask,
        target_mask=case.target_mask,
        metrics=metrics,
        caption=caption,
        confidence=float(result.confidence),
    )
    figure.savefig(os.path.join(FIGURES_DIR, f"worst_{case.name}.png"),
                   dpi=140, bbox_inches="tight")
    plt.show()

The best pairs, from the same records and the same rule inverted: highest IoU
first. Both cells write their panels into `results/figures/` as well as drawing
them inline, so a run on Colab leaves the images behind for the report.

In [ ]:
import json as _json
import os

import matplotlib.pyplot as plt

from evaluation.cases import cases_from_manifest
from evaluation.metrics import evaluate_mask
from visualization.benchmark import plot_benchmark_pair

manifests = [os.path.join(DATA_ROOT, "splits", f"{name}.json")
             for name in ("train", "validation", "test")]
cases, seen = [], set()
for path in manifests:
    if os.path.isfile(path):
        for case in cases_from_manifest(path):
            if case.name not in seen:
                seen.add(case.name)
                cases.append(case)

records = []
for case in cases:
    path = os.path.join(RESULTS_DIR, "samples", f"{case.name}.json")
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as handle:
            records.append(_json.load(handle))

scored = sorted((record for record in records if record.get("iou") is not None),
                key=lambda record: float(record["iou"]), reverse=True)
best = scored[:PAIR_FIGURES]
print(f"drawing the {len(best)} best of {len(scored)} scored pairs")

FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

by_name = {case.name: case for case in cases}
for record in best:
    case = by_name[str(record["sample_id"])]
    result = pipeline.segment_object(case.reference_image, case.reference_mask,
                                     case.target_image)
    metrics = evaluate_mask(result.predicted_mask, case.target_mask) if case.scorable else None
    caption = (f"{case.name} | {case.meta.get('category')} | "
               f"scale: {case.meta.get('scale_condition')} | "
               f"identity: {case.meta.get('identity_level')}")
    figure = plot_benchmark_pair(
        reference_image=case.reference_image,
        reference_mask=case.reference_mask,
        target_image=case.target_image,
        similarity_map=result.similarity_map,
        candidate_region=result.candidate_region,
        predicted_mask=result.predicted_mask,
        target_mask=case.target_mask,
        metrics=metrics,
        caption=caption,
        confidence=float(result.confidence),
    )
    figure.savefig(os.path.join(FIGURES_DIR, f"best_{case.name}.png"),
                   dpi=140, bbox_inches="tight")
    plt.show()

## 9. Provenance

Every run records the seed, the device, the package versions, the model ids and
the dataset identity, so a figure can be traced back to exactly what produced
it. The JSON is written next to the results.

In [ ]:
import json as _json
import os

provenance_path = os.path.join(RESULTS_DIR, "provenance.json")
with open(provenance_path, "r", encoding="utf-8") as handle:
    provenance = _json.load(handle)

for key in sorted(provenance):
    value = provenance[key]
    if isinstance(value, (dict, list)):
        print(f"{key}:")
        print("  " + _json.dumps(value, indent=2)[:2000].replace("\n", "\n  "))
    else:
        print(f"{key}: {value}")

## 10. Save back to Drive

Nothing in this notebook assumes it is running on Colab, but when it is, the
results are copied out of the ephemeral runtime into Drive so they survive a
disconnection. Locally the results already sit under `RESULTS_DIR`.

In [ ]:
import os
import shutil

target = RESULTS_DIR
if IN_COLAB and not os.path.abspath(RESULTS_DIR).startswith(os.path.abspath("/content/drive")):
    drive_target = "/content/drive/MyDrive/one-shot-results"
    print(f"copying {RESULTS_DIR} -> {drive_target}")
    shutil.copytree(RESULTS_DIR, drive_target, dirs_exist_ok=True)
    target = drive_target
else:
    print("results already outside the ephemeral runtime")

print("artefacts in", target)
for name in sorted(os.listdir(target)):
    path = os.path.join(target, name)
    if os.path.isfile(path):
        print(f"  {name:24s} {os.path.getsize(path) / 1024.0:8.1f} KB")
    else:
        print(f"  {name + os.sep:24s} {len(os.listdir(path)):8d} entries")